# Agentic AI Applications — Masterclass Part 1 of 3: Foundations & Orchestration

This is the second masterclass in the **AI Engineering series**. The first covered RAG. This one covers what comes after RAG — agents: the systems where an LLM doesn't just answer a question, it decides what to do, takes actions, observes results, and decides again.

Three notebooks in this masterclass:

| | Notebook | Scope |
|---|---|---|
| **N1** | **`02a_agents_foundations_orchestration.ipynb`** *(this one)* | Single-agent fundamentals → memory → planning → multi-agent patterns → DeepAgents |
| N2 | `02b_agents_protocols_coding.ipynb` | MCP, A2A, coding agents (Claude Code / Antigravity patterns) |
| N3 | `02c_agents_eval_serving_ops.ipynb` | Evaluation, inference & serving, monitoring, guardrails, maintenance |

## What this notebook covers

**Part A — What an agent is (§1-4)**: the ReAct loop, plan-and-execute, reflection, tool design, tool failures and retry.

**Part B — Memory, routing, planning, human-in-the-loop (§5-9)**: the four-layer memory model, when to route between models/tools/agents, the `write_todos` planning trick, LangGraph interrupts for human approval.

**Part C — Sync vs async, multi-turn (§10-11)**: when to reach for async, how multi-turn conversations actually keep state.

**Part D — Multi-agent orchestration (§12-17)**: when one agent isn't enough (the 15× cost rule), the supervisor and swarm patterns with working code, Anthropic's research-system architecture, **DeepAgents** as a batteries-included harness, custom handoffs.

## Stack (2026 defaults)

- **Python 3.11+**, `langchain` v1.x + `langchain-core`, `langgraph` v1.x
- **`langgraph-supervisor`**, **`langgraph-swarm`** for the multi-agent patterns
- **`deepagents`** as the batteries-included harness (Claude Code architecture, generalized)
- **`pydantic`** v2 for tool I/O at API boundaries (LangGraph state itself uses `TypedDict`)
- Provider SDKs: `langchain-openai`, `langchain-anthropic`, `langchain-google-genai`
- The code in this notebook runs offline using small stubs where APIs would normally be called. Real-world calls are shown in markdown.

## How to read this notebook

Every section has the same shape: short story → concept → why it matters → working code → tradeoffs → interview note. You can read top-to-bottom or jump to a section.

## A note on what we *don't* cover here

- Protocols (MCP, A2A) — in N2
- Coding agents specifically — in N2  
- Agent evaluation — in N3
- Inference, serving, observability, guardrails — in N3
- Maintenance and ops loop — in N3

## Setup


In [1]:
# Install the libraries you'll need. Comment out what you already have.
# !pip install -q langchain langchain-core langgraph langgraph-supervisor langgraph-swarm \
#                deepagents langchain-openai langchain-anthropic pydantic

# Optional: keys only needed for real LLM calls. This notebook works without them.
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

import warnings
warnings.filterwarnings("ignore")
print("Setup cell — install above if needed. All demos in this notebook run offline.")


Setup cell — install above if needed. All demos in this notebook run offline.


---
# Part A: What an Agent Is

## §1 — From chatbots to agents: the loop

A chatbot takes a message and produces a reply. That's a *function*, even if it's a smart one. An **agent** is something else: it takes a goal, decides what to do, takes an action, observes the result, and decides again — until the goal is reached or something stops it.

The difference isn't sophistication. It's the **loop**. Anything with this shape is an agent:

```
   ┌──────────────────────────────────┐
   │           goal arrives            │
   └─────────────────┬─────────────────┘
                     ▼
        ┌────────────────────────┐
        │  LLM decides next step │ ◄────┐
        └────────────┬───────────┘      │
                     │                  │
              ┌──────┴──────┐           │
              ▼             ▼           │
        [tool call]    [final answer]   │
              │             │           │
              ▼             ▼           │
        execute tool       done         │
              │                         │
              ▼                         │
        observation                     │
              └─────────────────────────┘
```

That's it. The whole field — ReAct, plan-and-execute, reflection, multi-agent, DeepAgents — is variations on this loop.

### Why the loop matters more than the LLM

A common mistake: thinking "agents = smarter LLM." False. The same LLM, *with* a loop and *with* tools, can solve problems that no amount of single-shot prompting can solve, because:

1. **It can take actions in the world.** Read files, query databases, send emails, write code. A chatbot can only output text.
2. **It can observe results.** Did the API call work? What did the database return? Was the test green? Single-shot prompting can't react to anything.
3. **It can self-correct.** Wrong answer → realize from the observation → try again. Chatbots can't.
4. **It can compose.** Five tools used in sequence produces behavior that no single tool could produce alone.

### ReAct: Reasoning + Acting (Yao et al. 2022)

The original agent loop. Interleave **Thought → Action → Observation**, repeat until done.

```
Thought:     "User wants weather in Delhi for tomorrow."
Action:      get_weather(city="Delhi", date="2026-05-20")
Observation: {"high": 38, "condition": "clear"}
Thought:     "I have enough to answer."
Final:       "Delhi tomorrow: high 38°C, clear."
```

**One thing to clear up**: in 2022, ReAct was implemented by parsing free-form text outputs ("Action:" prefix, regex extraction). In 2026, ReAct is implemented via **native tool-calling APIs** (OpenAI, Anthropic, Gemini all emit structured tool calls). The pattern is the same; the implementation is cleaner, faster, and more reliable. **Do not implement text-parsing ReAct in 2026.** Use native tool calling.

### Strengths of ReAct

- Trivially simple — fits in 20 lines
- Transparent — every step shows up as Thought / Action / Observation
- Works for dynamic tasks where you can't pre-plan

### Weaknesses

- **Local optima**: greedy step-by-step. No global view.
- **Infinite loops**: agent calls the same tool over and over. Need a step cap.
- **Cost**: N steps = N LLM calls. 10-step task on Claude Opus = real money.
- **Brittle on long horizons**: 30+ steps and the context gets noisy.

That's why §2 (planning) and Part D (multi-agent) exist.


In [2]:
# A minimal ReAct loop in ~40 lines, no LLM API needed.
# We fake the LLM with a rule-based router so the notebook runs anywhere.

from typing import Callable

# --- 1. Tools ---
def get_weather(city: str, date: str) -> dict:
    """Stub tool. Real one would call OpenWeather / WeatherAPI."""
    fake_db = {"Delhi": {"high": 38, "condition": "clear"},
               "Bangalore": {"high": 28, "condition": "rainy"}}
    return fake_db.get(city, {"error": "unknown city"})

def book_meeting(title: str, time: str) -> dict:
    """Stub tool. Real one would call Google Calendar."""
    return {"confirmed": True, "meeting_id": "abc-123", "title": title, "time": time}

TOOLS: dict[str, Callable] = {"get_weather": get_weather, "book_meeting": book_meeting}

# --- 2. Fake LLM (would be ChatAnthropic / ChatOpenAI in production) ---
def fake_llm_decide(history: list) -> dict:
    """Look at the conversation so far, decide: call a tool or answer."""
    last = history[-1]["content"].lower()
    # Naive routing — a real LLM would do this via tool-calling API.
    if "weather" in last and "delhi" in last and not any("get_weather" in str(m) for m in history):
        return {"type": "tool_call", "name": "get_weather", "args": {"city": "Delhi", "date": "2026-05-20"}}
    if "book" in last and "meeting" in last and not any("book_meeting" in str(m) for m in history):
        return {"type": "tool_call", "name": "book_meeting",
                "args": {"title": "Standup", "time": "2026-05-20T09:00"}}
    return {"type": "final", "content": "[LLM would synthesize a final answer from the history above]"}

# --- 3. The loop ---
def react_loop(user_query: str, max_steps: int = 10, verbose: bool = True) -> str:
    history = [{"role": "user", "content": user_query}]
    for step in range(max_steps):
        decision = fake_llm_decide(history)
        if decision["type"] == "final":
            if verbose: print(f"[step {step}] FINAL: {decision['content']}")
            return decision["content"]
        # Tool call path
        name, args = decision["name"], decision["args"]
        if verbose: print(f"[step {step}] ACTION: {name}({args})")
        try:
            result = TOOLS[name](**args)
        except Exception as e:
            result = {"error": f"{type(e).__name__}: {e}"}
        if verbose: print(f"[step {step}] OBSERVATION: {result}")
        history.append({"role": "assistant", "content": f"calling {name}({args})"})
        history.append({"role": "tool", "name": name, "content": str(result)})
    return "[max steps reached]"

# Try it
react_loop("What's the weather in Delhi tomorrow?")
print()
react_loop("Book a meeting called Standup tomorrow at 9am")


[step 0] ACTION: get_weather({'city': 'Delhi', 'date': '2026-05-20'})
[step 0] OBSERVATION: {'high': 38, 'condition': 'clear'}
[step 1] FINAL: [LLM would synthesize a final answer from the history above]

[step 0] ACTION: book_meeting({'title': 'Standup', 'time': '2026-05-20T09:00'})
[step 0] OBSERVATION: {'confirmed': True, 'meeting_id': 'abc-123', 'title': 'Standup', 'time': '2026-05-20T09:00'}
[step 1] FINAL: [LLM would synthesize a final answer from the history above]


'[LLM would synthesize a final answer from the history above]'

**Production form of the same loop.** In real code with native tool calling, the LLM emits a structured `tool_call` block; your runtime executes the tool and appends a `tool_result` message; the LLM sees it on the next turn. The skeleton:

```python
# Real-world equivalent — pseudocode using Anthropic SDK
from anthropic import Anthropic
client = Anthropic()
messages = [{"role": "user", "content": query}]
for _ in range(MAX_STEPS):
    resp = client.messages.create(model="claude-sonnet-4-6", tools=TOOLS_SCHEMA,
                                  messages=messages, max_tokens=2048)
    if resp.stop_reason == "end_turn":
        return resp.content[-1].text
    # Else: tool_use. Execute the tool, append tool_result, continue.
    for block in resp.content:
        if block.type == "tool_use":
            result = TOOLS[block.name](**block.input)
            messages.append({"role": "assistant", "content": resp.content})
            messages.append({"role": "user", "content": [
                {"type": "tool_result", "tool_use_id": block.id, "content": str(result)}
            ]})
```

That's a complete production ReAct agent. Every framework — LangChain `create_agent`, LangGraph's `create_react_agent`, OpenAI Agents SDK, DeepAgents — is some variation of this loop with extras.

## §2 — Plan-and-Execute and Reflection

ReAct is greedy. It picks one step, executes, observes, picks the next. That's perfect when you can't predict what comes next, but wasteful when you can. Two alternative loops fix the two main weaknesses of ReAct.

### Plan-and-Execute

**Idea**: an expensive planner LLM produces a full plan upfront. A cheap executor model runs each step. Replanner kicks in only on failure.

```
[Planner LLM (expensive)]  →  ordered plan or DAG
                                    │
                                    ▼
[Executor (cheap)]  →  step 1  →  step 2  →  step 3
                          │
                     failure?  →  [Replanner] → new plan
```

**Wins**: cost — you spend the big model's tokens once, not N times. **Parallelism** — independent steps in a DAG can run concurrently (`LLMCompiler`). **Auditability** — the plan is a visible artifact.

**Loses**: real tasks usually can't be fully pre-planned. Bug-fixing, exploratory research, conversation — all need the agent to react to what it sees mid-task. Plan-and-Execute is for **knowable workflows**, not exploratory ones.

**Variants**:
- **LLMCompiler** — plan as a DAG, run independent nodes in parallel.
- **ReWOO** (Reason Without Observation) — plan uses placeholders (`#E1`, `#E2`) for tool outputs; the worker fills them in. Cuts roundtrips because the planner doesn't have to wait for each observation.

### Reflection (Self-Critique)

**Idea**: an actor generates an answer; a critic evaluates it; the actor revises based on the critique. Stop when the critic is satisfied or max iterations hit.

```
[Actor]  ──generate──►  draft answer
   ▲                        │
   │                        ▼
   └──revise── [Critic]  ──critique──►  if "good", stop; else loop
```

**Wins**: catches errors the actor can't see in one shot. Especially powerful when there's a clear success criterion (code compiles, tests pass, schema validates, regex matches).

**Loses**: oscillation — actor and critic can ping-pong without converging. Always set a hard `max_iterations`. Cost — each iteration is 2× the LLM cost (gen + critique).

**Variants**:
- **Self-refine** (same model critiques itself — cheaper, but blind spots overlap)
- **Actor-Critic** (different models — better, costs more)
- **Reflexion** (lessons from past failures stored in episodic memory — covered in §5)

### Tradeoffs at a glance

| | ReAct | Plan-and-Execute | Reflection |
|---|---|---|---|
| Setup cost | Low | Medium | Medium |
| Per-task cost | High (N LLM calls) | Medium (1 plan + N cheap executor calls) | High (2× per iteration) |
| Parallelizable | No | Yes (DAG variant) | No |
| Best for | Dynamic / exploratory tasks | Known workflows, batch tasks | Quality-critical with eval signal |
| Worst at | Long horizons, cost | Tasks where plan changes mid-flight | Tasks without clear success criterion |

### Decision rule

> **Default to ReAct.** It's the simplest baseline that handles dynamic tasks. Wrap in **Plan-and-Execute** when the plan is stable and steps can run in parallel (research 50 papers, refactor 20 files). Add **Reflection** as a layer on top of either when there's a clear success criterion. **Never** add reflection without a `max_iterations` cap.

> **Interview note.** When asked "ReAct or plan-and-execute?" — the answer is never "one or the other." It's: "ReAct for exploration, plan-and-execute when the plan is knowable, often together: a top-level plan-and-execute with ReAct in each step where exploration is needed."


In [4]:
# Reflection pattern — actor + critic with a stop criterion.
# We fake the LLM with rule-based stubs so it runs offline.

def fake_actor(query: str, prior_critique: str = "") -> str:
    """Stub: pretend to be an LLM generating an answer (and revising if given a critique)."""
    if not prior_critique:
        return "Refunds usually take a few days."  # weak first draft
    if "specific" in prior_critique.lower() or "vague" in prior_critique.lower():
        return "Refunds are processed within 5-7 business days for all orders."  # revised
    return "Refunds are processed within 5-7 business days, with EU customers entitled to a 14-day window under consumer rights law."

def fake_critic(answer: str, success_criterion: str) -> dict:
    """Stub critic. Returns {good: bool, critique: str}. Real one would be an LLM."""
    if "5-7" not in answer:
        return {"good": False, "critique": "Answer is too vague — needs specific timeline."}
    if "business days" in answer:
        return {"good": True, "critique": "Answer is specific and grounded."}
    return {"good": False, "critique": "Missing 'business days' qualification."}

def reflect_and_revise(query: str, success_criterion: str,
                       max_iters: int = 3, verbose: bool = True) -> str:
    answer = fake_actor(query)
    if verbose: print(f"[iter 0] DRAFT: {answer}")
    for i in range(1, max_iters + 1):
        c = fake_critic(answer, success_criterion)
        if verbose: print(f"[iter {i-1}] CRITIQUE: good={c['good']} — {c['critique']}")
        if c["good"]:
            if verbose: print(f"[iter {i-1}] ACCEPTED.")
            return answer
        answer = fake_actor(query, prior_critique=c["critique"])
        if verbose: print(f"[iter {i}] REVISED: {answer}")
    if verbose: print(f"[max_iters reached] FALLBACK to last draft.")
    return answer

print("=== Reflection pattern in action ===\n")
final = reflect_and_revise(
    query="How long do refunds take?",
    success_criterion="specific timeline with business-day qualifier",
)
print(f"\nFinal answer: {final}")


=== Reflection pattern in action ===

[iter 0] DRAFT: Refunds usually take a few days.
[iter 0] CRITIQUE: good=False — Answer is too vague — needs specific timeline.
[iter 1] REVISED: Refunds are processed within 5-7 business days for all orders.
[iter 1] CRITIQUE: good=True — Answer is specific and grounded.
[iter 1] ACCEPTED.

Final answer: Refunds are processed within 5-7 business days for all orders.


## §3 — Tool design: schemas, descriptions, parallel calls

A tool is a typed function the LLM can decide to invoke. **A poorly designed tool is the #1 source of agent failures** — the model can't fix a bad tool, but a well-designed tool lets the model recover from almost anything.

### The anatomy of a tool

Every tool has three parts:

1. **A name** the LLM uses to call it.
2. **A description** the LLM reads to decide *whether* to call it.
3. **A typed parameter schema** the LLM uses to format its arguments.

```python
{
  "name": "search_flights",
  "description": "Search for available flights by origin, destination, and date. "
                 "Returns a list of {flight_id, airline, departure, price}. "
                 "Use this when the user asks about flight availability or prices. "
                 "Does NOT book flights — use book_flight for that.",
  "parameters": {
    "type": "object",
    "properties": {
      "origin": {"type": "string", "description": "IATA airport code, e.g. BLR for Bangalore"},
      "destination": {"type": "string", "description": "IATA airport code"},
      "date": {"type": "string", "format": "date", "description": "YYYY-MM-DD"},
    },
    "required": ["origin", "destination", "date"],
  },
}
```

### Description quality matters more than name

The LLM **routes between tools by reading descriptions**, not names. A great name with a terrible description fails; a clear description with a meh name works fine. Three things make a description good:

1. **What it does** in one sentence.
2. **What it returns** (the shape, not just "the result").
3. **When NOT to use it** (the boundary against similar tools).

That last bullet is the most-missed. With 10+ tools, the LLM picks the wrong one *most often* because two tools have overlapping descriptions and no boundary. Always answer: "What's the closest other tool, and how is this one different?"

### Parameter types matter for reliability

| Type signal | Why |
|---|---|
| Use `enum` when values are constrained | LLM can't hallucinate an invalid value |
| Use `format: date` / `date-time` / `email` | Validators reject malformed input before the tool runs |
| Use Pydantic models for nested args | Schema generation + validation for free |
| Required vs optional fields | The LLM defaults optional fields, asks for required ones |
| Defaults baked into the schema | Reduces the cognitive load on the LLM |

### Parallel tool calls — the 2026 default

Modern APIs (Anthropic, OpenAI, Gemini) can emit **multiple tool calls in one turn** when they're independent. Your runtime should execute them in parallel and return all results together.

```
Bad (sequential):              Good (parallel):
get_weather("Delhi")           get_weather("Delhi")     ──┐
↓ wait                         get_weather("Bangalore") ──┤── all return together
get_weather("Bangalore")       get_weather("Mumbai")    ──┘
↓ wait
get_weather("Mumbai")          → 1 round-trip, not 3
```

Anthropic's research-system blog (June 2025) explicitly tells lead agents to use parallel tool calls when spawning subagents:

> *"You MUST use parallel tool calls for creating multiple subagents (typically running 3 subagents at the same time) at the start of the research, unless it is a straightforward query."*

In §10 we'll show the `asyncio.gather` pattern that makes this happen.

### Side effects and idempotency

Tools that *read* (search, fetch, query) are safe to retry. Tools that *write* (send, create, charge, delete) are not — retrying after a partial failure can cause double-sends, double-charges, duplicate database rows.

**Two ways to handle this:**

1. **Make the tool idempotent.** Accept an `idempotency_key` that the runtime sets per call; the underlying service deduplicates. (Stripe, AWS APIs work this way.)
2. **Require explicit approval before destructive calls.** The agent proposes the action; a human-in-the-loop (§9) confirms; only then does the runtime execute. The 2026 default for anything involving money, deletion, or external messaging.

### Tool selection at scale

With 50+ tools, the LLM's tool-selection accuracy drops sharply. Three fixes:

| Fix | How |
|---|---|
| **Tool retrieval** | Embed tool descriptions; on each turn, retrieve top-K relevant tools and only show those to the LLM |
| **Tool namespacing** | Group tools: `gmail.send`, `gmail.search`, `calendar.create`. The LLM picks the namespace first |
| **MCP servers** (N2) | Discovery becomes standardized; the LLM doesn't need to know all tools upfront |

> **Interview note.** A question that catches candidates: *"You have 100 tools, accuracy drops. What do you do?"* The wrong answer is "use a bigger model." The right answer is **tool retrieval**: embed descriptions, retrieve top-5 per query, only show those. Same trick as RAG, applied to tools.


In [ ]:
# Tool design in practice: Pydantic for schema, asyncio for parallel execution.
from pydantic import BaseModel, Field
from typing import Literal
import asyncio
import time

# --- 1. Tool with a typed schema ---
class GetWeatherArgs(BaseModel):
    """Get current weather for a city on a date."""
    city: str = Field(..., description="City name, e.g. 'Delhi' or 'Bangalore'")
    date: str = Field(..., description="Date in YYYY-MM-DD format")
    unit: Literal["celsius", "fahrenheit"] = Field("celsius", description="Temperature unit")

# Tool implementation. In LangChain, you'd decorate with @tool and the schema is auto-generated.
async def get_weather_async(args: GetWeatherArgs) -> dict:
    """Async tool. Simulates network latency."""
    await asyncio.sleep(0.5)  # pretend this is a real HTTP call
    fake_db = {"Delhi": {"high": 38, "low": 28, "condition": "clear"},
               "Bangalore": {"high": 28, "low": 19, "condition": "rainy"},
               "Mumbai": {"high": 33, "low": 26, "condition": "humid"}}
    data = fake_db.get(args.city, {"error": "unknown city"})
    data["date"] = args.date
    return data

# --- 2. Schema export — what the LLM sees ---
schema = GetWeatherArgs.model_json_schema()
print("Schema the LLM sees (truncated):")
import json
print(json.dumps({"name": "get_weather", "description": GetWeatherArgs.__doc__,
                  "parameters": schema}, indent=2)[:500])

# --- 3. Sequential vs parallel — the timing demonstration ---
async def run_sequential(cities):
    start = time.perf_counter()
    results = []
    for c in cities:
        r = await get_weather_async(GetWeatherArgs(city=c, date="2026-05-20"))
        results.append(r)
    return results, time.perf_counter() - start

async def run_parallel(cities):
    start = time.perf_counter()
    tasks = [get_weather_async(GetWeatherArgs(city=c, date="2026-05-20")) for c in cities]
    results = await asyncio.gather(*tasks)
    return results, time.perf_counter() - start

cities = ["Delhi", "Bangalore", "Mumbai"]

# Run both
loop = asyncio.new_event_loop()
seq_results, seq_time = loop.run_until_complete(run_sequential(cities))
par_results, par_time = loop.run_until_complete(run_parallel(cities))
loop.close()

print(f"\nSequential: 3 tools × 0.5s each = {seq_time:.2f}s")
print(f"Parallel:   3 tools concurrently  = {par_time:.2f}s")
print(f"Speedup:    {seq_time/par_time:.1f}x")
print(f"\nThis is why modern LLM APIs emit multiple tool calls per turn.")


## §4 — Tool failures and retry logic

Tools fail. Networks blip. Rate limits trip. Auth tokens expire. APIs change their response shapes. The question is not *whether* your tools fail — it's whether your agent recovers gracefully or spirals.

### Three error categories

| Category | Examples | Right response |
|---|---|---|
| **Transient** | 5xx errors, network timeouts, rate-limit 429s | **Retry with backoff** |
| **Permanent** | 400 bad-request, 404 not-found, validation error | **Don't retry. Tell the LLM what went wrong.** |
| **Programming bugs** | KeyError, TypeError, IndexError in your tool code | **Don't hide. Fix the tool.** |

Conflating these is the #1 source of broken agents. Retrying a 400 just hammers the API; not retrying a 5xx makes the agent give up too early.

### Exponential backoff

For transient errors, retry after waiting `base × 2^attempt` seconds. **Add jitter** (random 0-1s on top) to avoid the thundering-herd problem when many agents retry simultaneously.

```
attempt 1: wait 1s + jitter
attempt 2: wait 2s + jitter
attempt 3: wait 4s + jitter
give up after 3-5 attempts
```

After max attempts, **surface the failure to the LLM as an observation**. Don't silently swallow it — the LLM can decide whether to try a different tool, ask the user, or give up.

### Idempotency: the prerequisite for safe retries

Retrying a `GET` is safe. Retrying a `POST` is dangerous unless the API supports idempotency keys.

The pattern:
```python
import uuid
def send_email(to, subject, body, idempotency_key=None):
    key = idempotency_key or str(uuid.uuid4())
    return api.send(to=to, subject=subject, body=body, Idempotency_Key=key)
```

The first call sends. A retry with the same key — even if you don't know the first one succeeded — gets a deduplicated response. Stripe, AWS, Twilio, and most modern APIs support this. Use it.

### Surfacing errors to the LLM (not hiding them)

The biggest mistake: wrapping every tool in a try/except that returns `"Error"` and nothing else. The LLM can't recover from "Error" — but it can recover from a *specific* error message:

```python
try:
    result = tool.run(**args)
except RateLimitError:
    result = {"error": "RateLimitError", "retry_after_seconds": 30,
              "suggestion": "Wait or use a less frequent tool"}
except NotFoundError:
    result = {"error": "NotFound", "message": f"No record with id={args['id']}",
              "suggestion": "Verify the id is correct"}
except Exception as e:
    result = {"error": type(e).__name__, "message": str(e)}
```

A well-formed error observation lets the LLM reason about it. A bare "Error" causes infinite retries or premature failure.

### Fallbacks: when retries aren't enough

For critical paths, define a fallback tool. Primary `web_search` (paid, high quality) → fallback `duckduckgo_search` (free, ok quality). The LLM only sees the unified interface; the wrapper picks which backend to use.

### Putting it together

The production pattern for tool execution:

```
1. Validate args against schema           → permanent error if invalid
2. Apply idempotency key for writes
3. Call the tool with timeout
4. On transient failure → exponential backoff + jitter, up to N retries
5. On permanent failure → return structured error message
6. On success → return result
7. Always return something the LLM can parse
```


In [10]:
# Exponential backoff + structured error surfacing.
import time
import random

class RateLimitError(Exception): pass
class NotFoundError(Exception): pass

# A flaky tool — fails twice, succeeds the third time.
_call_count = {"n": 0}
def flaky_search(query: str) -> dict:
    _call_count["n"] += 1
    if _call_count["n"] < 3:
        raise RateLimitError(f"Rate limited on attempt {_call_count['n']}")
    return {"results": [f"result for '{query}'"], "count": 1}

# Retry wrapper with exponential backoff + jitter
def retry_with_backoff(fn, *args, max_attempts=4, base_delay=0.2, **kwargs):
    """Retry transient errors with exponential backoff. Permanent errors bubble up."""
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(*args, **kwargs)
        except RateLimitError as e:
            if attempt == max_attempts:
                # Out of retries — surface as structured error
                return {"error": "RateLimitError", "attempts": attempt,
                        "message": str(e), "suggestion": "Try again later or use a cached result"}
            delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.1)
            print(f"  attempt {attempt} failed ({e}); sleeping {delay:.2f}s")
            time.sleep(delay)
        except NotFoundError as e:
            # Permanent — don't retry, return structured error immediately
            return {"error": "NotFound", "message": str(e),
                    "suggestion": "Verify the input is correct"}

# Demo 1: transient failure that recovers
_call_count["n"] = 0
print("=== Transient failure (3 attempts, succeeds on 3rd) ===")
result = retry_with_backoff(flaky_search, query="LangGraph multi-agent")
print(f"Final result: {result}\n")

# Demo 2: permanent failure (don't retry)
def lookup_user(user_id: str) -> dict:
    raise NotFoundError(f"User {user_id} does not exist")

print("=== Permanent failure (no retry) ===")
result = retry_with_backoff(lookup_user, user_id="u999")
print(f"Final result: {result}\n")

# The agent sees the error message and decides what to do next.
print("Key insight: the LLM gets a STRUCTURED error, not a bare exception.")
print("It can read 'RateLimitError' or 'NotFound' and reason about next steps.")


=== Transient failure (3 attempts, succeeds on 3rd) ===
  attempt 1 failed (Rate limited on attempt 1); sleeping 0.29s
  attempt 2 failed (Rate limited on attempt 2); sleeping 0.49s
Final result: {'results': ["result for 'LangGraph multi-agent'"], 'count': 1}

=== Permanent failure (no retry) ===
Final result: {'error': 'NotFound', 'message': 'User u999 does not exist', 'suggestion': 'Verify the input is correct'}

Key insight: the LLM gets a STRUCTURED error, not a bare exception.
It can read 'RateLimitError' or 'NotFound' and reason about next steps.


---
# Part B: Memory, Routing, Planning, Human-in-the-Loop

## §5 — Memory taxonomy: working, short-term, long-term, episodic, semantic, procedural

LLMs are stateless. Every API call starts fresh — nothing carries over unless you put it in the prompt. **An agent that needs to do anything across more than a single turn needs to manage its own memory.** This is unintuitive and skipped by most tutorials.

Memory in production agents is not one thing — it's a hierarchy, just like in cognitive science.

### The six types you'll be asked about

| Type | What it holds | Lifetime | Where it lives | Cost |
|---|---|---|---|---|
| **Working / scratchpad** | Current task: thoughts, intermediate results, draft outputs | This task only | Context window (in the prompt) | Free, but eats tokens |
| **Short-term** | Recent conversation messages | Current session | Context window or in-memory cache | Free, but eats tokens |
| **Long-term** | Persistent facts, user preferences, past conversations | Indefinite | Vector DB + structured DB | Storage + retrieval cost |
| **Episodic** | Discrete past events with timestamp + outcome | Indefinite | Structured DB (Postgres, BigQuery) | Storage |
| **Semantic** | General knowledge / concepts (think: the corpus in a RAG system) | Indefinite | Vector DB | Same as RAG |
| **Procedural** | How to do things — learned routines, "skills" | Indefinite | Code + prompts, sometimes fine-tuned | Storage + maintenance |

Don't get lost in the labels. The thing to internalize is the **two axes** they vary on:

1. **Lifetime** — does this last for one turn, one session, or forever?
2. **Structure** — is this a chunk of text, a vector, a row in a table, or actual code?

### Working memory (scratchpad)

The LLM's "thinking out loud" within a single task. Holds intermediate calculations, the plan, partial drafts. **Lives in the context window** — usually as the running message history, or in a structured `state` dict in LangGraph.

```python
# A LangGraph state with scratchpad-style fields
class AgentState(TypedDict):
    messages: list           # the running message history
    plan: list[str]          # the agent's plan, populated by a planner node
    intermediate_results: dict   # whatever the agent calculates along the way
    draft_answer: str
```

The agent reads and writes the scratchpad on every step. When the task ends, it's gone.

### Short-term memory (conversation buffer)

The last N messages of the current chat session. Three strategies for managing it as it grows past the context budget:

| Strategy | How | Trade-off |
|---|---|---|
| **Buffer window** | Keep last N messages, drop the rest | Simple. May lose important early context |
| **Summary buffer** | Summarize older messages into one "summary so far" system message; keep recent N verbatim | Preserves gist, loses precise wording |
| **Token-bounded** | Track total tokens; drop oldest until under budget | Most precise control |

**Production default**: summary buffer once you cross ~4K tokens, with the summary regenerated every N turns. The 2026 LangChain middleware has a `SummarizationMiddleware` that does this automatically.

### Long-term memory

User-specific persistent facts: preferences, past interactions, learned patterns. **The structure is RAG, applied to a per-user index** rather than a corporate document corpus.

```
On every turn:
  1. Embed the user's message
  2. Retrieve top-K relevant memories for this user_id
  3. Inject them into the system prompt as context
  4. Run the agent
  5. Maybe consolidate new memories (see §6)
```

Privacy is critical: **always filter by `user_id`**. Don't reuse a single embedding index across users without strict per-user partitioning, or you'll leak data between accounts (see §23 in N3).

### Episodic memory

Structured records of specific events. Not text chunks — **typed rows** with timestamps and outcomes. Used for:

- **Reflexion-style learning**: store lessons from failures, retrieve them when similar tasks come up
- **Trajectory replay** during debugging  
- **Audit trails** for compliance

```python
class Episode(BaseModel):
    timestamp: datetime
    user_id: str
    task: str
    actions_taken: list[str]
    outcome: Literal["success", "failure", "partial"]
    lesson: Optional[str]   # what would the agent do differently next time?
```

### Semantic and procedural — the often-skipped pair

**Semantic memory** is shared general knowledge. In an enterprise agent, it's the company's documentation corpus — exactly what we built RAG over in the prior notebook. The line between "long-term memory" and "semantic memory" is fuzzy: long-term = user-specific, semantic = world/domain knowledge.

**Procedural memory** is *how to do things*. Procedures the agent has learned: how to file an expense report, how to escalate a ticket, the steps in a deployment workflow. In practice this lives as **prompt templates + code**, not vectors. Some teams fine-tune small models on procedural data to bake it in.

### The architectural mistake to avoid

Don't put everything in one giant vector store. Each memory type has a different access pattern:

- Working memory needs **read/write per step** → in-memory state.
- Short-term needs **append + window** → list with summarization.
- Long-term needs **similarity search filtered by user_id** → vector DB.
- Episodic needs **structured queries on timestamps and outcomes** → SQL or document DB.

Cramming all four into one vector store gives you slow, fuzzy retrieval for queries that should be exact lookups.


## §6 — Memory consolidation, retention, and privacy isolation

Storing everything is bad. Storing nothing is bad. Memory management = **deciding what's worth keeping**, and **getting rid of what isn't**.

### When to promote something to long-term memory

Not every message in a chat deserves to live forever in the user's memory store. Four promotion heuristics:

1. **Explicit signal** — user says "remember this" or "from now on, prefer X."
2. **Repetition** — same thing mentioned 3+ times.
3. **High-value event** — task completion, explicit correction, preference statement.
4. **LLM judgment** — at the end of a session, ask a small model "is anything here worth remembering long-term?"

The combination is strongest. Always store explicit signals. Use the LLM judge for the rest.

### Retention policy

| Memory type | Retention rule |
|---|---|
| Working | Discard at task end |
| Short-term | Drop after session expires (e.g. 24 hours of inactivity) |
| Long-term | Retain indefinitely, but tier: hot (recent + frequent) vs cold (archive) |
| Episodic | Retain by regulation: financial data 7 years, support tickets 1 year |

The cold-tier trick saves a lot of money at scale. Move memories older than 90 days from your hot vector index to a cheaper store (S3, archive table) and re-promote them on miss.

### The privacy isolation pattern

The single most important rule: **every retrieval against long-term or episodic memory MUST filter by user_id at the database level, not the application level.**

```python
# WRONG — application-level filter
memories = vector_db.search(query_embedding, k=5)
memories = [m for m in memories if m.user_id == current_user]  # leak risk

# RIGHT — database-level filter
memories = vector_db.search(
    query_embedding, k=5,
    filter={"user_id": current_user}
)
```

Why does the wrong version leak? Because k=5 from the unfiltered search might return 5 *other users'* documents. The application-level filter then returns an empty list — but a logging system or a tool that captured the unfiltered results has already seen them.

### Memory hallucination — a real failure mode

If your agent uses an LLM to summarize and store memories, it can hallucinate facts into the memory store. Three turns later, the same user gets a confidently wrong "I remember you mentioned X" — except they never did.

**Defenses**:
- Validate factual claims before writing (does this quote actually appear in the conversation?).
- Store memories with provenance: which message did this come from?
- Periodically audit by sampling memories and checking against source.

> **Interview trap.** Candidate proposes "the agent summarizes the conversation at the end and stores the summary as a memory." Ask: *what stops the summary from being wrong?* Right answer: validation against the source transcript, plus provenance tracking.

### Putting it together — a memory class


In [7]:
# A simple four-layer agent memory.
# Working + short-term are in-process. Long-term + episodic are simulated stores.

from dataclasses import dataclass, field
from datetime import datetime
from typing import Literal, Any

@dataclass
class Episode:
    timestamp: datetime
    user_id: str
    task: str
    outcome: Literal["success", "failure", "partial"]
    lesson: str | None = None

class FakeVectorDB:
    """Stub. Real implementations: Qdrant, Pinecone, Chroma — see RAG notebook §7."""
    def __init__(self):
        self._store: list[dict] = []
    def upsert(self, record: dict):
        self._store.append(record)
    def search(self, query: str, filter: dict, k: int = 3) -> list[dict]:
        # CRITICAL: filter is database-level, applied before similarity ranking.
        candidates = [r for r in self._store if all(r.get(k_) == v for k_, v in filter.items())]
        # Real implementation would rank by vector similarity; we just return all matches.
        return candidates[:k]

class AgentMemory:
    """Four-layer memory for an agent. Production would split short-term to Redis,
    long-term to a real vector DB, and episodic to Postgres."""

    def __init__(self, user_id: str, vector_db: FakeVectorDB):
        self.user_id = user_id
        self.vector_db = vector_db
        self.working: dict[str, Any] = {}          # per-task, in RAM, discarded at end
        self.short_term: list[dict] = []           # session buffer
        self.episodes: list[Episode] = []          # episodic (in real life: a SQL table)

    # --- Working memory ---
    def set_scratchpad(self, key: str, value: Any) -> None:
        self.working[key] = value
    def get_scratchpad(self, key: str) -> Any:
        return self.working.get(key)

    # --- Short-term ---
    def append_message(self, role: str, content: str) -> None:
        self.short_term.append({"role": role, "content": content})

    def trim_short_term(self, max_messages: int = 10) -> None:
        """Buffer-window strategy. Production: use summary-buffer."""
        if len(self.short_term) > max_messages:
            # Real version: summarize the dropped ones into a system message before dropping.
            self.short_term = self.short_term[-max_messages:]

    # --- Long-term ---
    def recall(self, query: str, k: int = 3) -> list[dict]:
        """ALWAYS filters by user_id at the DB layer — privacy critical."""
        return self.vector_db.search(query=query, filter={"user_id": self.user_id}, k=k)

    def consolidate(self, fact: str, source_message_id: str) -> None:
        """Promote a fact to long-term memory with provenance."""
        # In production: validate the fact, embed it, then upsert.
        self.vector_db.upsert({"user_id": self.user_id, "text": fact,
                               "source": source_message_id, "timestamp": datetime.now().isoformat()})

    # --- Episodic ---
    def record_episode(self, task: str, outcome: str, lesson: str | None = None) -> None:
        self.episodes.append(Episode(
            timestamp=datetime.now(), user_id=self.user_id,
            task=task, outcome=outcome, lesson=lesson,
        ))

    def find_similar_episodes(self, current_task: str, outcome_filter: str = "failure") -> list[Episode]:
        """Reflexion-style: find past failures on similar tasks to learn from."""
        return [e for e in self.episodes
                if e.outcome == outcome_filter and current_task[:5].lower() in e.task.lower()][:3]

# --- Demo: a session with memory consolidation ---
db = FakeVectorDB()
mem = AgentMemory(user_id="samarth_2026", vector_db=db)

# Working memory during one task
mem.set_scratchpad("plan", ["search papers", "summarize", "write notes"])
print(f"Scratchpad: {mem.get_scratchpad('plan')}")

# Short-term — conversation buffer
mem.append_message("user", "I prefer concise summaries")
mem.append_message("assistant", "Noted. I'll keep summaries short.")

# Consolidate to long-term (explicit user preference signal)
mem.consolidate(fact="User prefers concise summaries", source_message_id="msg_1")

# Record an episode
mem.record_episode(task="summarize ML paper", outcome="success", lesson=None)

# Later: a new turn comes in, the agent recalls
print(f"\nRecalled for user: {mem.recall(query='formatting preferences')}")
print(f"Past failures on similar task: {mem.find_similar_episodes('summarize new paper')}")

# Privacy check: a DIFFERENT user gets nothing
other = AgentMemory(user_id="someone_else", vector_db=db)
print(f"Other user's recall (should be empty): {other.recall(query='formatting preferences')}")


Scratchpad: ['search papers', 'summarize', 'write notes']

Recalled for user: [{'user_id': 'samarth_2026', 'text': 'User prefers concise summaries', 'source': 'msg_1', 'timestamp': '2026-05-25T05:45:03.787134'}]
Past failures on similar task: []
Other user's recall (should be empty): []


## §7 — Routing: model, tool, and agent routing

"Routing" in an agent system means choosing one of N options given the current state. Three flavors, all with the same shape:

| Routing type | Choices | Example |
|---|---|---|
| **Model routing** | Cheap vs. premium LLM | Trivial query → Haiku; complex reasoning → Opus |
| **Tool routing** | Which tool to call | Refund question → policy DB; weather question → API |
| **Agent routing** | Which subagent / specialist to invoke | Code change → coding agent; research → research agent |

All three are decisions made *during* the agent loop. The same routing primitives apply to all three.

### Three ways to implement routing

#### 1. The LLM itself routes (tool-call routing)

The simplest pattern, the 2026 default for tool routing. The LLM sees all tools in the prompt and emits a tool call. No separate router code.

```
[LLM with tools] → emits tool_call(name="get_weather", args={...})
```

**Wins**: zero extra latency; the LLM's intent is visible. **Loses**: with 50+ tools, accuracy drops (see §3); no way to constrain the choice from outside.

#### 2. Explicit classifier in front of the LLM

A small/cheap model (or a fine-tuned classifier) maps the user query to a category, and the application picks the model/tool/agent.

```
user query → classifier → category → app picks downstream LLM/tool/agent
```

**Wins**: deterministic, fast, cheap. You can train it on your data. **Loses**: another moving part; classifier must be kept in sync as categories evolve.

This is what "**Adaptive RAG**" is (covered in RAG notebook §13): a router decides whether to skip retrieval, do a single retrieval, or enter a corrective loop.

#### 3. Conditional edges in a graph (LangGraph)

A function on the agent's state returns the name of the next node. The graph executes accordingly.

```python
def route_by_intent(state):
    if state["intent"] == "refund": return "refund_agent"
    if state["intent"] == "billing": return "billing_agent"
    return "general_agent"

builder.add_conditional_edges("classifier", route_by_intent,
                              {"refund_agent": "refund_node",
                               "billing_agent": "billing_node",
                               "general_agent": "general_node"})
```

This is how multi-agent supervisors work — covered in §13.

### Model routing — the cost lever you'll be asked about

In 2026, the smartest single decision in cost optimization is **route easy queries to cheaper models**:

| Query type | Route to | Why |
|---|---|---|
| Trivial classification, simple lookup | Haiku 4.5 / GPT-4o-mini | ~1/10 the cost |
| Standard reasoning, RAG, single-agent work | Sonnet 4.6 / GPT-5.x | Sweet spot |
| Long-horizon planning, multi-agent leadership | Opus 4.7 / GPT-5 Pro | Worth the premium for the orchestrator |
| Code generation, complex reasoning | Sonnet 4.6 with extended thinking, or Opus | Quality matters most here |

The Anthropic research-system blog explicitly used **Opus as the lead agent and Sonnet as the subagents** — the lead does the harder thinking, subagents do the bounded fetches. They report this beats a single Opus call by 90.2% on research evals at meaningful but bounded cost.

> **Interview note.** *"How would you cut your agent's bill in half?"* The right first answer is "**model routing** — classify queries by complexity, send the easy ones to a smaller model." Caching and trimming come next, but routing usually wins biggest.

### A tactical anti-pattern: routing for the sake of routing

Adding a classifier *before* a capable LLM that could route itself is wasteful when:
- You only have 2-3 tools
- Latency budget is tight
- The classifier's accuracy isn't measurably better than the LLM's intrinsic tool selection

**Default: let the LLM route tools.** Add an explicit classifier when (a) you have 10+ tools, (b) you want cost control via model routing, or (c) you need an audit log of every routing decision.


In [11]:
# Routing in three flavors. We stub the LLM and the downstream targets.

# --- Flavor 1: Model routing by complexity ---
def route_to_model(query: str) -> str:
    """Route to a model tier based on query complexity heuristics.
    In production this is a small classifier; here we keyword-match."""
    q = query.lower()
    if len(q.split()) <= 5 and any(w in q for w in ["what", "when", "where", "who"]):
        return "haiku-4.5"       # cheap, fast
    if any(w in q for w in ["plan", "design", "compare across", "evaluate trade-offs"]):
        return "opus-4.7"        # premium reasoning
    return "sonnet-4.6"          # default workhorse

for q in [
    "what is RAG",
    "Plan a 3-month roadmap to ship a multi-agent customer support system",
    "Summarize this PR for me",
]:
    print(f"  '{q}' → {route_to_model(q)}")

# --- Flavor 2: Tool routing via classifier ---
def route_to_tool(query: str) -> str:
    """Tool routing via classifier. In production: an embedding classifier or fine-tuned model."""
    q = query.lower()
    if "refund" in q or "money back" in q: return "refund_policy_search"
    if "shipping" in q or "delivery" in q: return "shipping_tool"
    if "weather" in q: return "weather_tool"
    return "general_search"

print("\n--- Tool routing ---")
for q in ["How do I get a refund?", "When will my package arrive?", "Weather in Delhi today"]:
    print(f"  '{q}' → {route_to_tool(q)}")

# --- Flavor 3: Conditional edges (LangGraph-style) ---
# We build a tiny graph by hand. The real version uses StateGraph.add_conditional_edges.

class MiniGraph:
    def __init__(self):
        self.nodes: dict = {}
        self.edges: dict = {}
        self.conditional_edges: dict = {}
        self.entry = None
    def add_node(self, name, fn):
        self.nodes[name] = fn
    def add_edge(self, from_, to):
        self.edges[from_] = to
    def add_conditional_edges(self, from_, router_fn, mapping):
        self.conditional_edges[from_] = (router_fn, mapping)
    def run(self, state, verbose=True):
        node = self.entry
        path = [node]
        while node and node != "END":
            state = self.nodes[node](state)
            if node in self.conditional_edges:
                router_fn, mapping = self.conditional_edges[node]
                next_key = router_fn(state)
                node = mapping[next_key]
            else:
                node = self.edges.get(node, "END")
            path.append(node)
        return state, path

# Nodes
def classify(state):
    state["intent"] = route_to_tool(state["query"])  # reuse the same router
    return state
def refund_handler(state):  state["response"] = "Refunds take 5-7 business days."; return state
def shipping_handler(state): state["response"] = "Standard shipping: 3-5 business days in India."; return state
def general_handler(state): state["response"] = "Let me search general knowledge."; return state

g = MiniGraph()
g.add_node("classify", classify)
g.add_node("refund", refund_handler)
g.add_node("shipping", shipping_handler)
g.add_node("general", general_handler)
g.entry = "classify"
g.add_conditional_edges("classify", lambda s: s["intent"],
                        {"refund_policy_search": "refund", "shipping_tool": "shipping",
                         "weather_tool": "general", "general_search": "general"})
g.add_edge("refund", "END"); g.add_edge("shipping", "END"); g.add_edge("general", "END")

print("\n--- Conditional-edge routing ---")
for q in ["How do I get a refund?", "When will my package arrive?", "What's the meaning of life?"]:
    final, path = g.run({"query": q})
    print(f"  '{q}'")
    print(f"    path = {' → '.join(path)}")
    print(f"    response = {final['response']}")


  'what is RAG' → haiku-4.5
  'Plan a 3-month roadmap to ship a multi-agent customer support system' → opus-4.7
  'Summarize this PR for me' → sonnet-4.6

--- Tool routing ---
  'How do I get a refund?' → refund_policy_search
  'When will my package arrive?' → general_search
  'Weather in Delhi today' → weather_tool

--- Conditional-edge routing ---
  'How do I get a refund?'
    path = classify → refund → END
    response = Refunds take 5-7 business days.
  'When will my package arrive?'
    path = classify → general → END
    response = Let me search general knowledge.
  'What's the meaning of life?'
    path = classify → general → END
    response = Let me search general knowledge.


## §8 — Planning patterns: write_todos as context engineering

We covered plan-and-execute in §2 as one of the three core agent loops. This section goes deeper on a related but distinct idea: **planning as context engineering**, which is the trick that powers Claude Code and DeepAgents.

### The write_todos pattern (the Claude Code trick)

Here's the surprising thing. Claude Code's planning tool is, mechanically, a **no-op**: it doesn't execute anything. It's a tool the agent calls to write down a todo list, mark items complete, and adapt the plan. The "tool" stores the plan in the agent's own state.

So why does it work?

Because **the act of writing down a plan, in the context window, changes the agent's subsequent behavior.** Three effects:

1. **Anchoring**: subsequent reasoning is grounded against the written plan. The agent doesn't drift.
2. **Progress tracking**: marking items complete frees the agent to focus on what's left, instead of re-deriving what's done.
3. **Long-horizon coherence**: across 50+ turns, the plan stays visible. The agent doesn't lose the thread.

This is what "context engineering" means in 2026. **The plan isn't for the user — it's for the agent itself.** You're shaping its working memory.

DeepAgents bakes this in: every agent gets a `write_todos` tool. It's not optional. Combined with the filesystem (offload large results) and subagents (isolate sub-task context), it's how a single agent stays coherent over hour-long tasks.

### Three planning styles

| Style | Plan when | Replan when |
|---|---|---|
| **Plan-then-execute** | Once, upfront | Never (or on full failure) |
| **Interleaved** (write_todos) | Upfront, *and* updated after each step | Continuously |
| **Reactive** (pure ReAct, no plan) | Never | N/A |

**Plan-then-execute** is best for batch workflows where the plan is stable (refactor 50 files; run 10 fixed checks).

**Interleaved** is the 2026 default for long, complex, exploratory tasks. The Claude Code / DeepAgents style.

**Reactive** is best for short, dynamic tasks where pre-planning would be wasted (one-shot questions, simple tool chains).

### Plan revision

The planner needs to know when its plan is wrong. Three signals:

1. **Tool failures** — a step failed, can we route around it?
2. **Unexpected results** — observation contradicts the plan's assumption.
3. **New information** — discovered something that requires a different approach.

In practice, after each step, the agent calls `write_todos` again to update the plan based on what it just learned. This is *cheaper* than the user thinks — the model already has the full context loaded, the tool call is essentially free, and the explicit re-write keeps the plan fresh.

### Plan-and-execute as a DAG (LLMCompiler)

For genuinely independent steps, you don't need to execute them in sequence. Build a DAG, run independent branches in parallel:

```
research_topic_A ──┐
research_topic_B ──┼──→ aggregate ──→ write_report
research_topic_C ──┘
```

LLMCompiler does this automatically: the planner emits a DAG, the executor runs branches in parallel. **Anthropic's research system uses this pattern.** The lead agent spawns 3+ subagents in parallel (parallel tool calls), each researches one topic, all return to the lead which aggregates.

> **Interview note.** *"What's the difference between a plan-and-execute agent and DeepAgents?"* DeepAgents is a *harness*, not a pattern. It bakes in the **interleaved** planning style (`write_todos`), a filesystem for context offloading, subagents for isolation, and a Claude Code-style system prompt. The "plan" in DeepAgents is the running todo list, not a one-shot DAG.

### Failure modes of planning

| Failure | Cause | Fix |
|---|---|---|
| Plan is wrong, agent follows it anyway | No revision step | Force re-plan after every step (interleaved) |
| Plan oscillates (rewritten every step with no progress) | Critic too strict, or success criterion unclear | Add cooldown — don't re-plan unless progress stalled for 2+ steps |
| Plan is too granular (50 todo items) | Planner over-thinks | System prompt: "plan in ≤7 high-level steps" |
| Plan is too coarse (1 todo: "do everything") | Planner under-thinks | System prompt: "break into 3-7 concrete steps before executing" |

Most planning bugs are prompt bugs. Tune the planner's prompt before tuning the code.


In [12]:
# write_todos pattern: a "no-op" planning tool that shapes the agent's behavior.
# Same trick used by Claude Code and DeepAgents.

class TodoList:
    """The planning artifact. Lives in the agent's state, not in any external system."""
    def __init__(self):
        self.items: list[dict] = []

    def write(self, current_task: str | None, next_steps: list[str]) -> str:
        """The 'tool' the agent calls. Returns a string the LLM sees as the observation."""
        # Mark prior current_task as done
        for item in self.items:
            if item["status"] == "in_progress":
                item["status"] = "done"
        # Add new steps as pending
        for step in next_steps:
            self.items.append({"step": step, "status": "pending"})
        # Mark the current_task in_progress
        if current_task:
            for item in self.items:
                if item["step"] == current_task and item["status"] == "pending":
                    item["status"] = "in_progress"
                    break
        return self.render()

    def render(self) -> str:
        """Human-readable view. This is what the LLM sees in its context window."""
        lines = ["TODO list:"]
        for i, item in enumerate(self.items, 1):
            mark = {"pending": "[ ]", "in_progress": "[~]", "done": "[x]"}[item["status"]]
            lines.append(f"  {mark} {i}. {item['step']}")
        return "\n".join(lines)

# --- Simulate an agent using the planning tool ---
todos = TodoList()

# Step 1: agent plans upfront
print("=== Agent plans upfront ===")
print(todos.write(current_task=None, next_steps=[
    "Research recent LangGraph multi-agent patterns",
    "Identify supervisor vs swarm trade-offs",
    "Draft a comparison table",
    "Write a 200-word summary",
]))

# Step 2: agent starts the first item
print("\n=== Agent begins first step ===")
print(todos.write(current_task="Research recent LangGraph multi-agent patterns", next_steps=[]))

# Step 3: agent completes first, moves to second — and ADDS a step it didn't expect
print("\n=== Agent finishes step 1, adapts plan ===")
print(todos.write(
    current_task="Identify supervisor vs swarm trade-offs",
    next_steps=["Verify with the LangGraph 2026 release notes"]  # unplanned but needed
))

# Step 4: finishes everything
print("\n=== Final state ===")
todos.write(current_task=None, next_steps=[])
for item in todos.items:
    if item["status"] != "done":
        item["status"] = "done"
print(todos.render())

print("\nKey insight: the 'tool' does nothing the agent couldn't do internally.")
print("But by writing the plan into its own context window, the agent stays coherent.")
print("This is what 'context engineering' means — the plan is for the agent, not the user.")


=== Agent plans upfront ===
TODO list:
  [ ] 1. Research recent LangGraph multi-agent patterns
  [ ] 2. Identify supervisor vs swarm trade-offs
  [ ] 3. Draft a comparison table
  [ ] 4. Write a 200-word summary

=== Agent begins first step ===
TODO list:
  [~] 1. Research recent LangGraph multi-agent patterns
  [ ] 2. Identify supervisor vs swarm trade-offs
  [ ] 3. Draft a comparison table
  [ ] 4. Write a 200-word summary

=== Agent finishes step 1, adapts plan ===
TODO list:
  [x] 1. Research recent LangGraph multi-agent patterns
  [~] 2. Identify supervisor vs swarm trade-offs
  [ ] 3. Draft a comparison table
  [ ] 4. Write a 200-word summary
  [ ] 5. Verify with the LangGraph 2026 release notes

=== Final state ===
TODO list:
  [x] 1. Research recent LangGraph multi-agent patterns
  [x] 2. Identify supervisor vs swarm trade-offs
  [x] 3. Draft a comparison table
  [x] 4. Write a 200-word summary
  [x] 5. Verify with the LangGraph 2026 release notes

Key insight: the 'tool' does 

## §9 — Human-in-the-loop with LangGraph interrupts

Some agent actions cost real money or affect real people: sending an email, charging a card, deleting a file, deploying code. For these, you don't want the agent to "decide and execute" in one motion — you want it to **decide, pause, get human approval, then execute.**

This is human-in-the-loop (HITL). LangGraph's `interrupt` primitive makes it elegant.

### The mechanism

A node can call `interrupt(payload)` — the graph pauses, the payload is returned to the caller, and the agent's state is **checkpointed** to a durable store. Hours or days later, a separate process calls the graph again with a `Command(resume=human_input)` — the graph picks up where it left off, with the human's input substituted for the interrupt's return value.

```
[node: propose_action] → state has the proposed action
        │
        ▼
[node: human_approval]
    │  calls interrupt({"action": state["proposed_action"]})
    │  graph pauses here, state is persisted
    ▼
[outside the graph]
    front-end shows the action to a human
    human clicks "approve" or "reject"
    backend calls graph.invoke(Command(resume="approve"))
        │
        ▼
[node: human_approval resumes, decision is the resume value]
        │
        ▼
[node: execute_action] only runs if approved
```

The graph doesn't hold a thread or process while waiting. The state is **fully persisted**. A pause can last hours, days, even weeks — the graph just resumes when the human clicks the button.

### When to use HITL

| Trigger | Example |
|---|---|
| Destructive actions | Delete a file, drop a table, cancel an order |
| Money-moving actions | Send a payment, issue a refund, place an order |
| External messaging | Send an email, post to social media, message a customer |
| Sensitive content review | Agent's draft about a delicate topic before it goes out |
| High-stakes decisions | Approve a contract, file a legal document |
| Onboarding | Verify user identity, collect missing required data |
| Cost-bounded escalation | Agent has spent $X on retries — confirm before more |

### When NOT to use HITL

| Anti-pattern | Why |
|---|---|
| Every tool call requires approval | Defeats the agent's purpose; user fatigue → rubber-stamp approval |
| Approval on read-only actions | Adds latency, no risk to mitigate |
| Approval AFTER the side-effect has happened | The horse has bolted |

### The four interrupt patterns

1. **Approve / reject before action** — most common
2. **Edit the proposed action** — human modifies the agent's draft, then runs
3. **Switch to human handoff** — agent recognizes it's out of depth, escalates to a human operator
4. **Inject missing info** — agent paused because a field was missing; human fills it in

### Failure modes

| Issue | Mitigation |
|---|---|
| Stale interrupts (pause for too long, business state changes) | TTL on threads; re-validate before executing approved actions |
| Approval bypass via prompt injection | The interrupt is structural, not text-based — but the approved action's args still need validation |
| Notification fatigue | Batch approvals, use confidence threshold (only interrupt if confidence < X) |
| Human-in-the-loop becomes the bottleneck | Confidence-based: interrupt only on high-stakes OR low-confidence actions |

> **Interview note.** *"How would you let an agent send emails on a user's behalf?"* The answer interviewers want: **agent drafts, human approves; never auto-send.** Use LangGraph interrupts, persist drafts with a TTL, validate the recipient list against an allow-list before final send. Bonus points: confidence threshold so low-risk drafts (say, "reply with thanks" to a known contact) auto-send but anything novel goes through approval.


In [13]:
# Human-in-the-loop with LangGraph interrupts.
# This runs the graph TWICE: first to pause at the interrupt, then to resume.

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

# --- State ---
class EmailState(TypedDict):
    to: str
    subject: str
    body: str
    approved: bool
    status: str

# --- Nodes ---
def draft_email(state: EmailState) -> dict:
    """Agent drafts the email."""
    body = f"Hi,\n\nThis is an automated message regarding: {state['subject']}.\n\nThanks!"
    print(f"  [draft]   to={state['to']}  subject={state['subject']!r}")
    return {"body": body, "status": "drafted"}

def request_approval(state: EmailState) -> dict:
    """Pause for human approval. The interrupt payload is what the front-end sees."""
    print(f"  [approve] requesting approval for email to {state['to']}")
    decision = interrupt({
        "action_type": "send_email",
        "to": state["to"],
        "subject": state["subject"],
        "body": state["body"],
        "prompt": "Approve to send, or Reject to cancel.",
    })
    # When the graph resumes via Command(resume=...), `decision` is the resume value.
    approved = decision == "approve"
    print(f"  [approve] human said: {decision!r} → approved={approved}")
    return {"approved": approved}

def send_or_skip(state: EmailState) -> dict:
    if state["approved"]:
        # In real life: actually call the email API here.
        print(f"  [send]    EMAIL SENT to {state['to']}")
        return {"status": "sent"}
    print(f"  [send]    SKIPPED (not approved)")
    return {"status": "rejected"}

# --- Graph ---
g = StateGraph(EmailState)
g.add_node("draft", draft_email)
g.add_node("approve", request_approval)
g.add_node("send", send_or_skip)
g.add_edge(START, "draft")
g.add_edge("draft", "approve")
g.add_edge("approve", "send")
g.add_edge("send", END)

# Compile with a checkpointer — required for interrupts (state must persist across the pause).
graph = g.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "user-123-email-1"}}

# --- Step 1: invoke. The graph runs until the interrupt and pauses. ---
print("=== Phase 1: agent drafts and pauses for approval ===")
initial = {"to": "samarth@example.com", "subject": "Welcome", "body": "", "approved": False, "status": ""}
result = graph.invoke(initial, config=config)

# Inspect the interrupt — this is what your UI would show
print(f"\n  state after pause: {result}")
state_now = graph.get_state(config)
if state_now.interrupts:
    print(f"  interrupted on: {state_now.interrupts[0].value.get('action_type')}")
    print(f"  → would render approval UI to the user")

# --- Step 2: human approves. Hours could pass between Phase 1 and Phase 2. ---
print("\n=== Phase 2: human clicks 'Approve', graph resumes ===")
final = graph.invoke(Command(resume="approve"), config=config)
print(f"\n  final state: {final}")


=== Phase 1: agent drafts and pauses for approval ===
  [draft]   to=samarth@example.com  subject='Welcome'
  [approve] requesting approval for email to samarth@example.com

  state after pause: {'to': 'samarth@example.com', 'subject': 'Welcome', 'body': 'Hi,\n\nThis is an automated message regarding: Welcome.\n\nThanks!', 'approved': False, 'status': 'drafted', '__interrupt__': [Interrupt(value={'action_type': 'send_email', 'to': 'samarth@example.com', 'subject': 'Welcome', 'body': 'Hi,\n\nThis is an automated message regarding: Welcome.\n\nThanks!', 'prompt': 'Approve to send, or Reject to cancel.'}, id='67ee4814e4f9335850d9678c239a0538')]}
  interrupted on: send_email
  → would render approval UI to the user

=== Phase 2: human clicks 'Approve', graph resumes ===
  [approve] requesting approval for email to samarth@example.com
  [approve] human said: 'approve' → approved=True
  [send]    EMAIL SENT to samarth@example.com

  final state: {'to': 'samarth@example.com', 'subject': 'Welc

/opt/anaconda3/envs/dsprep/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


---
# Part C: Sync vs Async, and Multi-turn Conversations

## §10 — Sync vs async: when to reach for each

You don't need to deeply understand asyncio to build agents. You do need to understand **what each mode is for**, so you can make the right call when designing a serving layer (covered properly in N3 §7).

### The four execution modes

| Mode | What | When to use |
|---|---|---|
| **Sync (blocking)** | Call → wait → return | One agent, one user, simple chatbot |
| **Async** | Call → yield → other work → resume | Parallel tool calls; concurrent users; mixing slow I/O |
| **Streaming** | Server emits tokens / events as they happen | Chat UIs (TTFT matters); long agents (show progress) |
| **Background jobs** | Submit → return immediately → poll / webhook later | Long-running agents (research, code generation runs of minutes-to-hours) |

### Why async matters for agents specifically

Two cases where sync is *measurably* worse:

#### 1. Parallel tool calls

When an LLM emits three independent tool calls in one turn (`get_weather(Delhi)`, `get_weather(Mumbai)`, `get_weather(Bangalore)`), executing them sequentially means waiting 3× the per-call latency. With async (`asyncio.gather`), they run concurrently — total time ≈ max of the three.

We already saw the demo in §3. Result: 3× speedup on N=3, scales linearly with the number of independent tool calls.

#### 2. Concurrent users on one server

A sync FastAPI server with one worker handles one request at a time. While Agent A is waiting on a tool response, Agent B can't even start. With async, the server multiplexes — Agent A waits on its tool, Agent B's LLM call is in flight, Agent C's response is streaming back.

**At any meaningful production scale, your agent server is async.** It's not optional.

### When sync is fine

- **Dev / debugging** — sync is easier to reason about.
- **Notebooks** — Jupyter has its own event loop quirks. Sync demos run cleanly.
- **One-off scripts** — batch processing where you process items in a loop.

### Streaming — orthogonal to sync/async

Streaming is about *the response format*, not the execution model. You can stream from sync code (chunked HTTP) or async code (SSE, WebSockets). The thing that streaming buys you:

- **Time-to-first-token (TTFT)** drops from full latency to ~200ms — users feel the agent is "thinking" instead of frozen.
- **Progress visibility** on long agents — show which subagent is running, what tool was just called.
- **Cancellation** — user can hit stop mid-stream and abort the agent.

LangGraph supports three stream "modes":
1. `stream_mode="values"` — emit full state snapshots after each node.
2. `stream_mode="updates"` — emit only the state diff per node (lighter).
3. `astream_events()` — emit fine-grained events (tokens, tool calls, completions).

### Background jobs — for when latency budget is "minutes"

A research agent that hits 50 web pages and writes a report doesn't fit in a request-response. Pattern:

```
POST /agent/run  →  returns {"job_id": "abc"} immediately
                    backend kicks off the agent in a worker

GET /agent/status/abc  →  returns {"state": "running", "progress": 0.4}
GET /agent/status/abc  →  returns {"state": "done", "result": {...}}
```

LangGraph's checkpointer plus a job queue (Celery, RQ, Temporal) is the standard recipe. We cover this in N3.

> **Interview note.** *"Sync or async for a multi-user agent backend?"* Always async, plus streaming for the user-facing channel, plus background jobs for anything >30s. The 2026 stack: FastAPI (async) + LangGraph with PostgresSaver + Server-Sent Events for stream + a worker queue for long jobs.


In [14]:
# Two patterns of async that matter for agents:
# 1) Parallel tool calls within one agent turn
# 2) Concurrent agents sharing one event loop

import asyncio, time, random

async def fake_tool(name: str, delay: float) -> str:
    """Simulate a tool that takes `delay` seconds (e.g. a network call)."""
    await asyncio.sleep(delay)
    return f"{name} result"

# --- Pattern 1: parallel tool calls in one turn ---
async def one_agent_turn():
    """LLM emits three tool calls in one turn. Execute all in parallel."""
    start = time.perf_counter()
    results = await asyncio.gather(
        fake_tool("weather_delhi", 0.4),
        fake_tool("weather_mumbai", 0.4),
        fake_tool("weather_bangalore", 0.4),
    )
    elapsed = time.perf_counter() - start
    return results, elapsed

# --- Pattern 2: concurrent agents (multiple users on one server) ---
async def one_full_agent_run(user_id: str) -> dict:
    """Simulate a full agent: 2 LLM calls + 2 tool calls."""
    start = time.perf_counter()
    # Turn 1: LLM call (0.3s) + tool call (0.4s)
    await asyncio.sleep(0.3)
    await fake_tool(f"tool-{user_id}-1", 0.4)
    # Turn 2: LLM call (0.3s) + tool call (0.4s)
    await asyncio.sleep(0.3)
    await fake_tool(f"tool-{user_id}-2", 0.4)
    elapsed = time.perf_counter() - start
    return {"user": user_id, "elapsed": round(elapsed, 2)}

async def server_handling_many_users():
    """Three users hit the server at the same time."""
    start = time.perf_counter()
    results = await asyncio.gather(
        one_full_agent_run("alice"),
        one_full_agent_run("bob"),
        one_full_agent_run("carol"),
    )
    total = time.perf_counter() - start
    return results, total

# Run both demos
loop = asyncio.new_event_loop()

print("=== Pattern 1: parallel tool calls within one turn ===")
results, elapsed = loop.run_until_complete(one_agent_turn())
print(f"  3 tools × 0.4s each, run in parallel")
print(f"  total wall time: {elapsed:.2f}s  (would be 1.2s if sequential)")

print("\n=== Pattern 2: 3 concurrent agents on one async server ===")
results, total = loop.run_until_complete(server_handling_many_users())
for r in results:
    print(f"  user {r['user']}: agent took {r['elapsed']}s")
print(f"  server total wall time: {total:.2f}s")
print(f"  (would be ~4.2s if sync — 3 users × ~1.4s each, sequential)")

loop.close()
print("\nTakeaway: async lets ONE process handle parallel tool calls AND concurrent users.")
print("In a real FastAPI server, multiple agents run simultaneously on one event loop.")


=== Pattern 1: parallel tool calls within one turn ===


RuntimeError: Cannot run the event loop while another loop is running

## §11 — Multi-turn conversations: keeping state across turns

A "multi-turn conversation" is what every user actually wants. The LLM doesn't remember previous turns natively — every API call is stateless. So you have to manage the conversation yourself.

This sounds trivial. It mostly is. But there are three failure modes that catch every team at least once.

### What "managing state" actually means

Between turns, you persist:

1. **The message history** (or a summary of it).
2. **The agent's working state** — current plan, partial results, anything not in the messages.
3. **A pointer** (thread_id, conversation_id) that ties together this user's conversation.

In LangGraph, all three are handled by **checkpointers**. Pass a `thread_id` in the config; the graph reads + writes state to the checkpointer; on the next turn with the same thread_id, the state is there.

```python
config = {"configurable": {"thread_id": "user-123-conv-7"}}
result1 = graph.invoke({"messages": [user_msg_1]}, config=config)
# ... time passes ...
result2 = graph.invoke({"messages": [user_msg_2]}, config=config)
# result2 has access to everything from result1 — same thread_id.
```

In LangChain (without LangGraph), you compose a `RunnableWithMessageHistory` and pass a session ID; same idea, lower-level.

### Three failure modes

#### 1. Context overflow

Each turn appends to the history. By turn 20, the context is bloated. By turn 50, it's truncated and the agent loses early context.

**Fix**: summary-buffer strategy. Once total tokens cross a threshold (e.g. 4K), have a small LLM summarize the older messages into a single system message, drop the originals, keep the recent N verbatim.

```
[system: agent prompt]
[system: SUMMARY OF PRIOR CONVERSATION — "User wants to book flights, prefers cheapest..."]
[user: latest message]
[assistant: ...]
[user: latest message]
```

LangChain 2026 has `SummarizationMiddleware` that does this automatically.

#### 2. Stale state

Day 1: user says "my email is samarth@example.com". Day 30: email has changed. The agent re-uses the old value from memory.

**Fix**: TTL on facts; re-validate at action time. For anything user-stated, ask the user to confirm before acting on facts older than N days.

#### 3. Lost-thread bug

Bug class: user opens a new tab/device; the front-end forgets the thread_id; the agent starts from scratch. User says "as I mentioned earlier..." — the agent has no idea what was said earlier.

**Fix**: thread_id is server-side, tied to user + conversation. Don't trust the client to provide it. Front-end fetches the active thread_id from a server endpoint on app load.

### What goes in the system prompt vs the conversation

A common confusion: what's in the system prompt, what's in the running history?

| Goes in system prompt (re-injected every turn) | Goes in conversation history |
|---|---|
| Stable instructions ("you are a customer support agent") | The actual back-and-forth |
| User profile facts (preferences, role, language) | Tool calls and observations |
| Long-term memories retrieved this turn | Drafts and revisions |
| Available tools (handled by the tool-calling API) | The current task's scratchpad |

Mixing these up costs tokens and confuses the LLM. The system prompt should be **stable across the conversation**. Dynamic per-turn context (RAG retrievals, long-term memory) goes into a *separate* system message that gets refreshed each turn.

### The 2026 production pattern

```
[stable system prompt: identity + tools]
[dynamic system prompt: relevant memories + RAG context for this turn]
[summary of older conversation if any]
[recent N message exchanges, verbatim]
[user's new message]
```

That's it. Every turn, you reconstruct this. The agent emits its reply, tool calls happen, and the state goes back to the checkpointer.


In [15]:
# Multi-turn conversation with a checkpointer. The graph keeps state across calls.
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class ChatState(TypedDict):
    messages: Annotated[list, add]     # reducer: append, so we accumulate across turns
    turn_count: int

def chat_node(state: ChatState) -> dict:
    """Pretend this is a real LLM. We just print and respond."""
    last_user_msg = next((m for m in reversed(state["messages"]) if m["role"] == "user"), None)
    turn = state.get("turn_count", 0) + 1
    msg_count = len(state["messages"])
    # Stub response — a real LLM call would go here
    response = f"[turn {turn}] I see {msg_count} messages so far. You just said: {last_user_msg['content'][:50]}"
    print(f"  [chat_node] turn={turn}, total messages in state={msg_count}")
    return {
        "messages": [{"role": "assistant", "content": response}],
        "turn_count": turn,
    }

g = StateGraph(ChatState)
g.add_node("chat", chat_node)
g.add_edge(START, "chat")
g.add_edge("chat", END)
graph = g.compile(checkpointer=MemorySaver())   # critical: checkpointer persists state

# --- Turn 1 ---
config = {"configurable": {"thread_id": "user-samarth-conv-1"}}
print("=== Turn 1 ===")
out = graph.invoke({"messages": [{"role": "user", "content": "Hello, I want to learn about agents"}]},
                   config=config)
print(f"  state.turn_count = {out['turn_count']}")
print(f"  state.messages   = {len(out['messages'])} messages")

# --- Turn 2 (same thread_id) — state carries over ---
print("\n=== Turn 2 (same thread) ===")
out = graph.invoke({"messages": [{"role": "user", "content": "Specifically multi-agent ones"}]},
                   config=config)
print(f"  state.turn_count = {out['turn_count']}")
print(f"  state.messages   = {len(out['messages'])} messages  ← grew because reducer appends")

# --- Turn 3 ---
print("\n=== Turn 3 ===")
out = graph.invoke({"messages": [{"role": "user", "content": "And how to evaluate them"}]},
                   config=config)
print(f"  state.turn_count = {out['turn_count']}")

# --- A DIFFERENT thread_id: fresh state ---
print("\n=== A new thread_id — fresh state ===")
other_config = {"configurable": {"thread_id": "user-samarth-conv-2"}}
out = graph.invoke({"messages": [{"role": "user", "content": "Different conversation"}]},
                   config=other_config)
print(f"  state.turn_count = {out['turn_count']}  ← reset, this is a new thread")
print(f"  state.messages   = {len(out['messages'])} messages  ← starts fresh")

# --- Inspect history with time travel ---
print("\n=== Time travel on the first thread ===")
history = list(graph.get_state_history(config))
print(f"  {len(history)} checkpoints stored for the first conversation")
print(f"  Could fork from any past checkpoint to try an alternative path.")


=== Turn 1 ===
  [chat_node] turn=1, total messages in state=1
  state.turn_count = 1
  state.messages   = 2 messages

=== Turn 2 (same thread) ===
  [chat_node] turn=2, total messages in state=3
  state.turn_count = 2
  state.messages   = 4 messages  ← grew because reducer appends

=== Turn 3 ===
  [chat_node] turn=3, total messages in state=5
  state.turn_count = 3

=== A new thread_id — fresh state ===
  [chat_node] turn=1, total messages in state=1
  state.turn_count = 1  ← reset, this is a new thread
  state.messages   = 2 messages  ← starts fresh

=== Time travel on the first thread ===
  9 checkpoints stored for the first conversation
  Could fork from any past checkpoint to try an alternative path.


---
# Part D: Multi-Agent Orchestration

## §12 — Why multi-agent: when one agent isn't enough

Multi-agent is the answer to "why isn't my single ReAct agent working?" — but it's also the answer to a hundred problems that don't actually need it. **Know the cost before you reach for it.**

### Anthropic's 15× cost rule

From the June 2025 Anthropic engineering blog on their multi-agent research system:

> *Multi-agent systems use about 15× more tokens than chat. For these systems to make economic sense, the application has to involve tasks where the value of completing the task justifies the increased cost.*

Fifteen times. That's the number to memorize.

Why so much? Because each subagent:
- Has its own system prompt
- Runs its own ReAct loop
- Generates its own intermediate reasoning
- Returns its results to the lead, which generates more reasoning to integrate them

A query that would take one Sonnet call ($0.005) becomes a coordinated dance of one Opus lead + three Sonnet subagents, each making 5-10 tool calls = maybe $0.075. **You're paying for breadth and depth.**

So: don't reach for multi-agent unless you're delivering 15× more value than a single agent.

### When multi-agent genuinely beats single-agent

| Use case | Why multi-agent wins |
|---|---|
| **Research** — explore many angles in parallel | Parallel breadth — 3 subagents researching simultaneously is 3× faster (in wall-time) and finds more |
| **Distinct skill domains** — research + code + writing in one task | Specialization — each subagent has a focused prompt + tool set, fewer tool selection mistakes |
| **Cross-doc aggregation** — summarize 50 customer interviews | Scatter-gather is the natural pattern |
| **Quality-critical with eval signal** — code that must pass tests | Actor-critic, with the critic genuinely catching mistakes |
| **Context isolation needed** — subtasks generate a lot of noise | Each subagent has its own clean context window |

Anthropic reports their multi-agent research system **outperformed a single Opus agent by 90.2% on internal research evals.** That gap justifies the 15× cost — for that workload.

### When single-agent is the right call

| Use case | Why single-agent wins |
|---|---|
| Most chatbots and customer support | A well-instrumented single ReAct agent with good tools handles 95% of queries |
| Single-fact RAG | A retrieval pipeline + one LLM call is enough |
| Short-horizon tasks (5-10 steps) | Coordination overhead exceeds the value |
| Cost-sensitive | 15× is a lot — only justified for high-value tasks |
| Tasks without parallel-able subtasks | Sequential dependencies kill the speedup |

### The decision tree

```
Will this task complete in <10 LLM calls?
  YES → single ReAct agent. Done.
  NO  → continue
        │
Are there 3+ truly independent subtasks?
  NO  → single agent with planning (write_todos), maybe reflection
  YES → multi-agent. Continue to "which pattern?" (§13-15)
```

### Failure modes Anthropic explicitly called out

Worth memorizing — these come up in interviews verbatim:

1. **Spawning 50 subagents for simple queries** — fix: scaling rules embedded in the lead's system prompt
2. **Subagents duplicating each other's work** — fix: precise, non-overlapping task descriptions per subagent
3. **Endless web searches for nonexistent sources** — fix: max-tool-calls budget per subagent
4. **Subagents distracting each other with excessive updates** — fix: structured outputs only; no chatty messages between subagents

### Scaling rules embedded in prompts (Anthropic pattern)

From the blog:

> *Simple fact-finding: 1 agent, 3-10 tool calls*  
> *Direct comparison: 2-4 subagents, 10-15 tool calls each*  
> *Complex research: 10+ subagents, divided responsibilities*

These get **written into the lead agent's system prompt** as policy. The agent reads its own rules and bounds itself. Without this, the lead spawns hordes for trivial queries.

> **Interview note.** *"When would you use multi-agent vs single-agent?"* Lead with the cost rule (~15× more tokens). Then the value-of-task threshold. Then the patterns (research, distinct skills, cross-doc, quality-critical). The candidate who says "always use multi-agent, it's more powerful" loses points.


## §13 — Supervisor pattern

One coordinator agent routes work to specialists. The supervisor decides *which* agent handles the task; the specialists execute.

```
                ┌──────────────┐
                │  Supervisor  │
                └──────┬───────┘
            ┌────────┼────────┐
            ▼        ▼        ▼
       [Researcher][Coder][Writer]
            │        │        │
            └────────┼────────┘
                     ▼
              back to Supervisor
                     │
                     ▼
              final response
```

The supervisor sits in the middle of every transition. After a specialist finishes, control returns to the supervisor, which decides what to do next (call another specialist, finish, or escalate).

### When the supervisor is the right call

| Signal | Why supervisor |
|---|---|
| 3+ distinct skill domains (research, code, writing) | Specialization helps; clean routing |
| User talks to one persona (the "assistant") | Single voice; routing happens behind the scenes |
| Routing decisions need LLM reasoning | Hard-coded rules can't capture "which agent should answer this?" |
| Need audit trail of who did what | Supervisor's decisions are all visible in trace |

### When NOT to use a supervisor

| Anti-pattern | Better alternative |
|---|---|
| Strictly sequential pipeline (always A then B then C) | Linear graph, no supervisor needed |
| Single-step lookup | Plain function call or single agent |
| Peer-to-peer handoff where the user "talks to" a specialist directly | Swarm (§14) |
| When the supervisor needs domain expertise to route | Co-pilot pattern: have specialists run, then a final aggregator picks the best response |

### The library: `langgraph_supervisor`

Current 2026 API:

```python
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent

# Build specialists
researcher = create_react_agent(
    model=model,
    tools=[web_search, fetch_page],
    prompt="You research topics and return findings.",
    name="researcher",
)
writer = create_react_agent(
    model=model,
    tools=[],
    prompt="You write clean reports from research findings.",
    name="writer",
)

# Wrap them in a supervisor
supervisor = create_supervisor(
    agents=[researcher, writer],
    model=model,
    prompt=("Delegate research tasks to the researcher; then send findings to the writer. "
            "When the user's request is fully addressed, finish."),
).compile(checkpointer=checkpointer)
```

A `recursion_limit` on `invoke()` is essential — without it, a supervisor that mis-routes can loop forever. Default to 25; alarm if you hit it.

### The four production-bugs you'll see

| Bug | Symptom | Fix |
|---|---|---|
| Supervisor tries to do the work itself | Skips specialists | Give the supervisor zero specialist tools; only "route" or "finish" |
| Supervisor loops between specialists | Two agents each say "talk to the other one" | Stricter prompt + recursion_limit alarm |
| Supervisor's prompt grows unbounded | Token cost blows up | Trim handoff messages — `add_handoff_back_messages=False` |
| Routing accuracy decays over time | Wrong specialist picked too often | Build a routing-accuracy eval *before* shipping; gate prompt changes on it |

### Tradeoffs

| | Supervisor wins | Cost |
|---|---|---|
| Routing accuracy | High — single LLM with full context decides | Bottleneck — every transition goes through it |
| Auditability | Every routing decision logged | Latency — adds an LLM call per hop |
| Specialization | Each specialist has focused prompt + tools | More state to manage |
| Predictability | Decision logic in one place | "Routing creativity" can backfire — wrong specialist picked |


In [16]:
# Supervisor pattern. We show the production-API skeleton, then a runnable simulation.

supervisor_code = '''
# Production code — requires `langgraph-supervisor` and a real LLM.
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import MemorySaver

model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

def web_search(query: str) -> str:
    """Search the web. Returns top results."""
    return f"[results for {query}]"

def write_report(findings: list[str]) -> str:
    """Compose a final report from findings."""
    return "\\n".join(f"- {f}" for f in findings)

# Specialists
researcher = create_react_agent(
    model=model, tools=[web_search],
    prompt="You research topics. Use web_search up to 5 times. Return concise findings.",
    name="researcher",
)
writer = create_react_agent(
    model=model, tools=[],
    prompt="You write clean, well-structured reports from research findings.",
    name="writer",
)

# Supervisor — note the deliberately tight scope of its prompt
supervisor = create_supervisor(
    agents=[researcher, writer],
    model=model,
    prompt=("Coordinate two specialists: researcher (gathers info) and writer (composes reports). "
            "Always send the user's question to the researcher first, then pass findings to the writer. "
            "When the writer returns the final report, end."),
    output_mode="last_message",
).compile(checkpointer=MemorySaver())

# Run
result = supervisor.invoke(
    {"messages": [{"role": "user", "content": "Compare LangGraph supervisor vs swarm patterns"}]},
    config={"configurable": {"thread_id": "demo-1"}, "recursion_limit": 25},
)
print(result["messages"][-1].content)
'''
print(supervisor_code)

# --- Runnable simulation: shows the routing pattern without real LLM calls ---
print("\n=== Simulated supervisor (no LLM, just the routing pattern) ===\n")

def simulated_supervisor_route(task: str, completed: list[str]) -> str:
    """Stub. A real supervisor LLM would read state and emit a handoff tool call."""
    if "researcher" not in completed:
        return "researcher"
    if "writer" not in completed:
        return "writer"
    return "FINISH"

def researcher_agent(task: str) -> str:
    return f"Findings on '{task}': supervisor routes through one orchestrator; swarm has peer handoffs."

def writer_agent(findings: str) -> str:
    return f"Report:\n  {findings}\n  Trade-offs: supervisor=routing accuracy; swarm=lower latency."

task = "Compare LangGraph supervisor vs swarm"
completed: list[str] = []
state = {"task": task, "findings": "", "report": ""}

for step in range(5):  # recursion bound
    next_node = simulated_supervisor_route(task, completed)
    print(f"  [supervisor] → routing to: {next_node}")
    if next_node == "FINISH":
        print(f"  [supervisor] FINISH\n")
        break
    if next_node == "researcher":
        state["findings"] = researcher_agent(task)
        print(f"  [researcher] {state['findings']}")
        completed.append("researcher")
    elif next_node == "writer":
        state["report"] = writer_agent(state["findings"])
        print(f"  [writer] {state['report']}")
        completed.append("writer")

print("Final state:")
print(f"  report: {state['report']}")



# Production code — requires `langgraph-supervisor` and a real LLM.
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import MemorySaver

model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

def web_search(query: str) -> str:
    """Search the web. Returns top results."""
    return f"[results for {query}]"

def write_report(findings: list[str]) -> str:
    """Compose a final report from findings."""
    return "\n".join(f"- {f}" for f in findings)

# Specialists
researcher = create_react_agent(
    model=model, tools=[web_search],
    prompt="You research topics. Use web_search up to 5 times. Return concise findings.",
    name="researcher",
)
writer = create_react_agent(
    model=model, tools=[],
    prompt="You write clean, well-structured reports from research findings.",
    name="writer",
)

# Supervisor — note the deliberately tight scop

## §14 — Swarm pattern: peer-to-peer handoffs

The supervisor sits in the middle of every transition. **The swarm eliminates the middle.** Each agent has the ability to hand off directly to a peer; whichever agent is "active" handles the user's turn.

```
        ┌──────────────┐
        │   Triage     │◄────┐
        └───┬──────────┘     │
            │ handoff        │ handoff back
            ▼                │
        ┌──────────────┐     │
        │  Specialist  │─────┘
        └──────────────┘

       Whoever is "active" handles the current user message.
       Handoffs are direct, agent-to-agent.
```

The user experience is different: they "talk to" whichever agent is currently active. In customer support: triage takes the call, hands off to billing, billing solves it and hands back. The user feels they're talking to "a person" who may have transferred them.

### When the swarm is the right call

| Signal | Why swarm |
|---|---|
| User naturally "talks to" specialists directly | Persona handoff matches the mental model |
| Latency-sensitive, supervisor adds too much hop overhead | One fewer LLM call per transition |
| Domain experts know who to hand off to better than a coordinator | Distributed decision-making |
| Customer support, sales call routing, multi-persona dialog | The canonical use cases |

### When the swarm is wrong

| Signal | Why supervisor instead |
|---|---|
| You need audit-trail of routing decisions | Swarm decisions are distributed; harder to trace |
| Routing logic is complex (depends on many state fields) | Centralize it in the supervisor |
| You're new to multi-agent | Supervisor is simpler to reason about — start there |

> **Default**: start with supervisor. Graduate to swarm only when latency data shows the supervisor is the bottleneck **and** your agents rarely misroute. The supervisor is easier to debug — every transition shows in one trace.

### The library: `langgraph_swarm`

```python
from langgraph_swarm import create_swarm, create_handoff_tool
from langgraph.prebuilt import create_react_agent

# Each agent has handoff tools pointing at its peers
triage_to_billing = create_handoff_tool(
    agent_name="billing",
    description="Hand off to the billing agent when the user has billing or refund questions.",
)
billing_to_triage = create_handoff_tool(
    agent_name="triage",
    description="Hand back to triage when the billing question is resolved.",
)

triage_agent = create_react_agent(
    model=model, tools=[triage_to_billing],
    prompt="You triage incoming questions. Hand off billing questions to billing.",
    name="triage",
)
billing_agent = create_react_agent(
    model=model, tools=[billing_to_triage, get_invoice, issue_refund],
    prompt="You handle billing. After resolving, hand back to triage.",
    name="billing",
)

swarm = create_swarm(
    agents=[triage_agent, billing_agent],
    default_active_agent="triage",
).compile(checkpointer=checkpointer)
```

### Under the hood: handoffs are `Command` objects

When an agent calls a handoff tool, the tool returns a `Command` that tells LangGraph to redirect execution to a different node:

```python
return Command(
    goto="billing",                # which agent to switch to
    graph=Command.PARENT,           # navigate in the parent graph, not the subagent's
    update={"messages": [tool_message]},  # what state to update
)
```

That's why "swarm" works — there's no central node deciding routes; the handoff tools themselves contain the navigation.

### Three failure modes specific to the swarm

| Bug | Symptom | Fix |
|---|---|---|
| Ping-pong handoffs | Agents keep handing back and forth | Add stay-active guidance in prompts; set recursion_limit |
| Lost context across handoffs | New agent doesn't know what was said | Default `create_handoff_tool` passes full message history — confirm this is on |
| Tool overlap | Two agents both have `send_email` — unclear who acts | Strict per-agent tool sets; one tool, one owner |

### Quick comparison

| | Supervisor | Swarm |
|---|---|---|
| Topology | Hub-and-spoke | Mesh |
| Per transition | 2 LLM calls (specialist + supervisor) | 1 LLM call (next agent) |
| Routing accuracy | Higher (full context, one decider) | Lower (distributed decisions) |
| Latency | Higher | Lower |
| Debuggability | Easier | Harder without good tracing |
| User mental model | "I'm talking to one assistant" | "I got transferred to a specialist" |
| **Start here** | Yes | No — only graduate when needed |


In [17]:
# Swarm pattern — same structure: production-API skeleton, then a runnable simulation.

swarm_code = '''
# Production code — requires `langgraph-swarm` and a real LLM.
from langgraph_swarm import create_swarm, create_handoff_tool
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import MemorySaver

model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

def get_invoice(invoice_id: str) -> dict:
    return {"id": invoice_id, "amount": 99.0, "status": "paid"}

def issue_refund(invoice_id: str) -> dict:
    return {"refunded": True, "invoice": invoice_id}

# Handoff tools — these are what create the swarm topology
triage_to_billing = create_handoff_tool(
    agent_name="billing",
    description="Hand off to billing for invoice, refund, or payment questions.",
)
billing_to_triage = create_handoff_tool(
    agent_name="triage",
    description="Hand back to triage when the billing matter is resolved.",
)

triage = create_react_agent(
    model=model,
    tools=[triage_to_billing],
    prompt="You triage incoming questions. Hand billing-related questions to the billing agent.",
    name="triage",
)
billing = create_react_agent(
    model=model,
    tools=[billing_to_triage, get_invoice, issue_refund],
    prompt="You handle billing. After resolving, hand back to triage with a summary.",
    name="billing",
)

swarm = create_swarm(
    agents=[triage, billing],
    default_active_agent="triage",
).compile(checkpointer=MemorySaver())

result = swarm.invoke(
    {"messages": [{"role": "user", "content": "Refund invoice INV-42 please"}]},
    config={"configurable": {"thread_id": "demo-swarm-1"}, "recursion_limit": 25},
)
'''
print(swarm_code)

# --- Runnable simulation showing the topology ---
print("\n=== Simulated swarm (no LLM, just the handoff mechanics) ===\n")

class HandoffCommand:
    """Stub of langgraph.types.Command for a swarm-style handoff."""
    def __init__(self, goto: str, message: str):
        self.goto = goto
        self.message = message

def triage_agent(message: str) -> HandoffCommand | str:
    """Decides whether to handle in-place or hand off."""
    if any(w in message.lower() for w in ["refund", "invoice", "billing", "charge"]):
        return HandoffCommand(goto="billing", message=f"User says: {message}")
    return "Triage: I'm not sure, can you clarify?"

def billing_agent(message: str) -> HandoffCommand | str:
    """Handles billing tasks; hands back when done."""
    if "refund" in message.lower():
        return HandoffCommand(goto="triage", message="Billing: I refunded the invoice and notified the user.")
    return "Billing: I can help with refunds, invoices, charges."

# Simulate the swarm runtime
AGENTS = {"triage": triage_agent, "billing": billing_agent}
active = "triage"  # default_active_agent
user_message = "Refund invoice INV-42 please"
print(f"  [user] {user_message}")
print(f"  [active = {active}]")

for hop in range(5):  # recursion limit
    handler = AGENTS[active]
    response = handler(user_message)
    if isinstance(response, HandoffCommand):
        print(f"  [{active}] → HANDOFF to {response.goto}: {response.message}")
        active = response.goto
        user_message = response.message  # pass context with the handoff
    else:
        print(f"  [{active}] {response}")
        break

print(f"\n  Final active agent: {active}")
print("Notice: no central supervisor. Agents decide handoffs themselves via handoff tools.")



# Production code — requires `langgraph-swarm` and a real LLM.
from langgraph_swarm import create_swarm, create_handoff_tool
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import MemorySaver

model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

def get_invoice(invoice_id: str) -> dict:
    return {"id": invoice_id, "amount": 99.0, "status": "paid"}

def issue_refund(invoice_id: str) -> dict:
    return {"refunded": True, "invoice": invoice_id}

# Handoff tools — these are what create the swarm topology
triage_to_billing = create_handoff_tool(
    agent_name="billing",
    description="Hand off to billing for invoice, refund, or payment questions.",
)
billing_to_triage = create_handoff_tool(
    agent_name="triage",
    description="Hand back to triage when the billing matter is resolved.",
)

triage = create_react_agent(
    model=model,
    tools=[triage_to_billing],
    prompt="You triage i

## §15 — Hierarchical and Scatter-Gather + Anthropic's research system

Beyond the two-tier supervisor and swarm, two more patterns matter for production. Both are about **partitioning work** so that single-agent constraints (context window, tool-selection paralysis) don't dominate.

### Scatter-Gather (a.k.a. Map-Reduce)

A planner decomposes the task into N independent subtasks. N workers run in parallel. A gatherer aggregates results.

```
            [ Planner / Lead ]
              ┌─────┼─────┐
              ▼     ▼     ▼
         [Worker 1][2][3]    (parallel)
              │     │     │
              └─────┼─────┘
                    ▼
              [ Aggregator ]
                    │
                    ▼
                response
```

**When**: subtasks are truly independent. "Research these 10 startups", "Process this batch of 50 documents", "Compare these 5 products on price + warranty + reviews."

**Wins**: near-linear speedup in wall-time. Each worker has clean context. Naturally combines with parallel tool calls.

**Loses**: aggregation is often the hard part. Five workers each return 2KB of text; aggregator needs to deduplicate, resolve conflicts, prioritize. Pure scatter-gather only works when results compose simply.

### Hierarchical

A supervisor of supervisors. The top layer routes between *teams*; each team has its own supervisor routing between *specialists*.

```
            [ Top supervisor ]
              ┌────┴────┐
              ▼         ▼
     [Research team] [Code team]
       sup → specialists  sup → specialists
```

**When**: 10+ specialists, with natural team groupings. Without hierarchy, a single supervisor choosing between 15 specialists makes routing mistakes constantly.

**Loses**: each layer is a hop, latency adds up. Two-LLM-call cost per transition becomes 4-LLM-call cost per top-level decision.

### Anthropic's research system — the canonical scatter-gather case study

From their June 2025 blog. The architecture:

```
                  [User query]
                       │
                       ▼
              [Customer Support Agent]
              (clarifies, queues task)
                       │
                       ▼
                [ Lead Researcher ]
                (Opus 4 — the expensive thinking)
                       │
                       ├── plans the breakdown
                       ├── stores plan in memory (because the task is too big for context)
                       │
                       ▼ (spawns 3-10 subagents in parallel via parallel_tool_calls)
                ┌──────┼──────┐
                ▼      ▼      ▼
        [Subagent 1] [2]   [N]
        (Sonnet 4 — bounded workers)
        each: 3-10 tool calls
                │      │      │
                └──────┼──────┘
                       ▼
                [ Lead aggregates ]
                       │
                       ▼
                [ Copywriter Agent ]
                (produces the final report)
                       │
                       ▼
                  [PDF, delivered]
```

Several engineering choices worth internalizing:

1. **Opus as lead, Sonnet as subagents.** Asymmetric cost — the lead does the hard thinking; subagents do the bounded fetches. Result: 90.2% improvement over single Opus.

2. **Plan is stored in memory, not just context.** The task is large enough that the plan would get truncated; explicit persistence keeps the lead aligned across the hour-long task.

3. **Scaling rules in the lead's prompt.** Simple query → 1 agent, 3-10 calls. Comparison → 2-4 subagents, 10-15 calls each. Complex research → 10+ subagents with divided responsibilities. **The lead reads its own rules to bound itself.**

4. **Parallel tool calls mandatory.** The lead must spawn subagents in parallel — sequential spawning would defeat the whole point. The system prompt enforces this.

5. **Strict subagent prompts.** Each subagent gets: an objective, an output format, tool guidance, and clear task boundaries. Vague instructions caused duplicate work in early iterations.

### Pattern selection

| Pattern | Best for | Gotcha |
|---|---|---|
| Supervisor | 3-7 specialists, clear domains | Supervisor becomes bottleneck |
| Swarm | 2-3 personas, user talks to specialists | Hard to debug; routing mistakes |
| Scatter-gather | Independent parallel subtasks | Aggregation is the hard part |
| Hierarchical | 10+ specialists, team groupings | Latency stacks per layer |

> **Interview trap.** Candidate proposes "flat" multi-agent (every agent can call every other) for production customer support. Push back: emergent behavior is unpredictable, token costs unbounded, hard to debug. Flat is a research pattern, not a production one. Use supervisor + specialist subgraphs + HITL for approvals — boring is right.


In [18]:
# Scatter-gather pattern, modeled after Anthropic's research system.
# Lead agent decomposes a query, spawns N subagents IN PARALLEL via asyncio,
# then aggregates. We use stubs for the LLM calls so it runs offline.

import asyncio
from dataclasses import dataclass

@dataclass
class SubagentTask:
    """The structured task description the lead gives each subagent.
    Anthropic's blog: 'each subagent needs an objective, output format,
    tool guidance, and clear task boundaries.'"""
    objective: str
    output_format: str
    tool_guidance: str
    boundaries: str
    max_tool_calls: int = 8

async def subagent(task: SubagentTask) -> dict:
    """A bounded subagent. Simulates 3-10 tool calls + a structured return."""
    print(f"  [subagent] starting: {task.objective}")
    # Simulate 3-8 tool calls
    tool_calls_made = min(task.max_tool_calls, 5)
    await asyncio.sleep(0.3)  # simulate the work
    findings = f"Researched '{task.objective}' with {tool_calls_made} tool calls"
    print(f"  [subagent] done: {findings}")
    return {"objective": task.objective, "findings": findings, "tool_calls": tool_calls_made}

# Anthropic-style scaling rules embedded in the lead's policy.
def scale_to_query_complexity(query: str) -> int:
    """How many subagents to spawn, based on the query."""
    q = query.lower()
    # 1 / 2-4 / 10+ pattern from the Anthropic blog
    if any(w in q for w in ["compare", "versus", "vs"]):
        return 3   # comparison → 2-4 subagents
    if any(w in q for w in ["research", "deep dive", "comprehensive"]):
        return 6   # complex research → 10+ subagents (capped at 6 for the demo)
    return 1       # simple fact-finding → 1 agent

async def lead_agent(user_query: str) -> str:
    """The expensive Opus-style lead. Plans, spawns in parallel, aggregates."""
    print(f"[lead] received: {user_query}")

    # Step 1: scaling rule
    n = scale_to_query_complexity(user_query)
    print(f"[lead] complexity rule says: {n} subagent(s)")

    # Step 2: plan — decompose into N specific tasks (this would be an LLM call in production)
    tasks = [
        SubagentTask(
            objective=f"Aspect {i+1} of: {user_query}",
            output_format="Structured findings: facts + sources",
            tool_guidance="web_search up to 5 times, fetch_page for primary sources",
            boundaries=f"Focus only on aspect {i+1}; do NOT explore aspects {[j+1 for j in range(n) if j!=i]}",
            max_tool_calls=8,
        )
        for i in range(n)
    ]

    # Step 3: SPAWN IN PARALLEL — the Anthropic blog explicitly requires this
    print(f"[lead] spawning {n} subagent(s) in parallel via asyncio.gather")
    results = await asyncio.gather(*(subagent(t) for t in tasks))

    # Step 4: aggregate (would be an LLM call; we just concatenate)
    print(f"[lead] aggregating {len(results)} subagent results")
    report = " | ".join(r["findings"] for r in results)
    total_tool_calls = sum(r["tool_calls"] for r in results)
    return f"REPORT (lead aggregated, total {total_tool_calls} subagent tool calls): {report}"

# Run two queries to see the scaling rule in action
loop = asyncio.new_event_loop()
print("=== Simple query — should use 1 subagent ===")
out = loop.run_until_complete(lead_agent("What is RAG?"))
print(f"  result: {out}\n")

print("=== Comparison — should use 2-4 subagents ===")
out = loop.run_until_complete(lead_agent("Compare LangGraph supervisor vs swarm"))
print(f"  result: {out}\n")

print("=== Complex research — should use many subagents ===")
out = loop.run_until_complete(lead_agent("Deep dive into agentic AI evaluation frameworks"))
print(f"  result: {out}")
loop.close()


=== Simple query — should use 1 subagent ===


RuntimeError: Cannot run the event loop while another loop is running

## §16 — DeepAgents: the batteries-included harness

DeepAgents is, in one sentence, **"Claude Code's architecture, extracted and generalized."** Released open-source by LangChain in mid-2025, it took off because the existing answer ("write your own LangGraph harness") was too much work for too many teams.

### What problem it solves

Single-agent ReAct (`create_agent`) is a "shallow" agent — it's just a tool-calling loop. Production-grade agents like Claude Code, Cursor, and Devin have four extra capabilities that turn shallow agents into deep ones:

| The four pillars | What it does |
|---|---|
| **Detailed system prompt** | Teaches the agent how to behave in specific situations — far longer and more specific than typical prompts |
| **Planning tool** (`write_todos`) | Lets the agent maintain a running plan in its own state. Crucially, the tool is a no-op — it's context engineering |
| **Filesystem access** | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`. The filesystem is the agent's working memory beyond the context window |
| **Subagent spawning** | A `task` tool that creates a specialized subagent with its own isolated context |

DeepAgents bundles all four into a single `create_deep_agent()` call. Same building blocks as `create_agent`, but with the harness pre-wired.

### The minimal API

```python
from deepagents import create_deep_agent

def web_search(query: str) -> str:
    """Search the web."""
    return f"[results for {query}]"

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",     # any LangChain chat model
    tools=[web_search],                       # your custom tools
    system_prompt="You are a research assistant.",  # added to the built-in DeepAgents prompt
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Research multi-agent eval frameworks"}]
})

# Result includes the full message history AND any files the agent wrote
print(result["messages"][-1].content)
print(list(result.get("files", {}).keys()))   # files the agent created during the task
```

Three lines to get an agent with planning, filesystem, subagents, and context management. That's why it took off.

### When to reach for DeepAgents

| Use case | DeepAgents |
|---|---|
| Long-horizon research tasks (hours of work) | ✓ ideal |
| Complex code generation with file edits | ✓ ideal |
| Multi-step analytical workflows | ✓ ideal |
| Short Q&A | ✗ overkill — use `create_agent` |
| Pure routing chatbots | ✗ overkill — use `create_agent` |
| Custom non-ReAct topology | ✗ drop to raw LangGraph |

### How it stacks against `create_agent` and raw LangGraph

```
                         Cost / complexity →
   raw LangGraph    create_agent (LangChain)    DeepAgents
   ▲                       ▲                       ▲
   custom graphs           simple ReAct agent      batteries-included harness
   any topology            single-agent loop       planning + FS + subagents + prompt
   most work               little work             least work, when task is a "project"
```

Pick:
- **Raw LangGraph** when the agent loop isn't ReAct-shaped (custom multi-agent, weird state machines).
- **`create_agent`** when you have a ReAct agent with a handful of tools, nothing exotic.
- **DeepAgents** when the task is a *project* — long horizons, file edits, planning, delegation.

The three compose: a LangGraph CompiledStateGraph can be passed in as a subagent to a Deep Agent. So you can use DeepAgents as the top-level harness and drop into raw LangGraph where needed.

### The "filesystem" — it's the agent's extra brain

This is the underappreciated bit. When an agent's context approaches the limit, DeepAgents writes intermediate results to a (virtual) file and reads them back later. Pluggable backends:

| Backend | Use |
|---|---|
| In-memory state | Default, fastest |
| Local disk | Persistent across runs |
| LangGraph store | Cross-thread persistence (long-term memory) |
| Sandboxes (Modal, Daytona, Deno) | Isolated code execution — covered in N2 coding agents |
| Composite routing | Combine multiple — e.g. memory + sandbox |

For coding agents (N2), the sandbox backend is the production default — the agent can write and execute code without affecting the host.

### Tradeoffs

| | DeepAgents wins | Cost |
|---|---|---|
| Setup time | 3 lines to a capable agent | Less control over the loop |
| Context management | Filesystem + subagents + summarization built in | More moving parts |
| Long-horizon coherence | Excellent — planning tool keeps the agent on track | Probably overkill for short tasks |
| Provider-agnostic | Works with any LangChain chat model | Same as any LangChain dependency |

> **Interview note.** *"What's the difference between DeepAgents, LangGraph, and `create_agent`?"* All three are layers in the same stack. **LangGraph** is the runtime (state machine, checkpointer, durable execution). **`create_agent`** is a minimal ReAct harness on top of LangGraph. **DeepAgents** is an opinionated harness on top of `create_agent` with planning, filesystem, subagents, and a detailed system prompt baked in. Use DeepAgents when the task is long and complex; drop down only when needed.


In [ ]:
# DeepAgents — production API skeleton (requires `deepagents` + an LLM).

deepagents_code = '''
# Production code — requires `pip install deepagents` and a real LLM.
from deepagents import create_deep_agent

def web_search(query: str) -> str:
    """Search the web for a query. Returns top results as text."""
    # Real: hook up Tavily, SerpAPI, Brave Search, etc.
    return f"[results for {query}]"

def fetch_page(url: str) -> str:
    """Fetch a URL and return its main text content."""
    return f"[page content for {url}]"

# Three lines to a research agent with planning, filesystem, and subagents built in.
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[web_search, fetch_page],
    system_prompt=(
        "You are a thorough research assistant. "
        "Plan before you act. Use the filesystem for any intermediate notes. "
        "Spawn subagents for independent research sub-topics."
    ),
)

# Invoke
result = agent.invoke({
    "messages": [{"role": "user", "content":
                  "Research multi-agent eval frameworks. Compare LangSmith, Phoenix, Ragas."}]
})

# Final answer
print(result["messages"][-1].content)

# Files the agent wrote during the run — useful for debugging
print("Files created:", list(result.get("files", {}).keys()))
# Common: 'plan.md', 'findings/langsmith.md', 'findings/phoenix.md', 'final_report.md'
'''
print(deepagents_code)

print("\n=== What you would see during a run ===\n")

# A trace-like illustration of what DeepAgents does internally
trace = [
    ("tool_call", "write_todos", {"current_task": None, "next_steps": [
        "Search for LangSmith agent eval docs",
        "Search for Phoenix agent eval docs",
        "Search for Ragas agent eval docs",
        "Compare features in a table",
        "Write the final report",
    ]}),
    ("tool_call", "task", {"description": "Research LangSmith agent eval", "subagent": "researcher"}),
    ("tool_call", "task", {"description": "Research Phoenix agent eval", "subagent": "researcher"}),
    ("tool_call", "task", {"description": "Research Ragas agent eval", "subagent": "researcher"}),
    ("note", "above 3 task tool calls would run IN PARALLEL — that's the point", None),
    ("tool_call", "write_file", {"path": "findings/langsmith.md", "content": "..."}),
    ("tool_call", "write_file", {"path": "findings/phoenix.md", "content": "..."}),
    ("tool_call", "write_file", {"path": "findings/ragas.md", "content": "..."}),
    ("tool_call", "write_todos", {"current_task": "Compare features in a table", "next_steps": []}),
    ("tool_call", "read_file", {"path": "findings/langsmith.md"}),
    ("tool_call", "read_file", {"path": "findings/phoenix.md"}),
    ("tool_call", "read_file", {"path": "findings/ragas.md"}),
    ("tool_call", "write_file", {"path": "final_report.md", "content": "comparison table + analysis"}),
    ("final", "Done. Final report in final_report.md", None),
]
for kind, name, args in trace:
    if kind == "tool_call":
        args_short = str(args)[:80]
        print(f"  [{kind}] {name}({args_short})")
    elif kind == "note":
        print(f"  ⤳ {name}")
    else:
        print(f"  [{kind}] {name}")


## §17 — Handoffs: state transfer, context compression, custom handoff tools

Whether in a supervisor (specialist → supervisor → next specialist) or in a swarm (peer → peer), handoffs are how state moves between agents. Get them right and the system feels coherent; get them wrong and each transition feels like a fresh conversation.

### What "handoff" actually means

A handoff is two things at once:

1. **Control transfer** — execution moves from agent A to agent B.
2. **State transfer** — what agent A knew, agent B now also knows (or, importantly, *doesn't*).

LangGraph implements both via the `Command` object — `Command(goto=...)` is the control transfer, and the optional `update={...}` is the state transfer.

### The default behavior

When you create a handoff tool with `create_handoff_tool(agent_name="X")`:

- The full message history of the supervisor / current agent is passed along.
- A ToolMessage indicating successful handoff is appended.
- The receiving agent sees everything.

That's fine for short conversations. It breaks at scale.

### Three problems with the default

#### 1. Context bloat

By the time the third specialist sees the conversation, it includes:
- The original user message
- The supervisor's reasoning
- The first specialist's full ReAct trace
- The supervisor's next routing decision
- The second specialist's full ReAct trace
- The supervisor's next routing decision

That's 5,000+ tokens of "stuff that doesn't matter to me." The third specialist now reasons over a polluted context.

#### 2. Privacy / scope creep

Specialist B doesn't need to see Specialist A's internal scratchpad reasoning. Sometimes A handled sensitive data (a customer's account number) that B shouldn't have. The default "pass everything" leaks.

#### 3. Specialist confusion

Specialist B's tools are described in B's system prompt. But the message history is full of A's tool calls. The model can get confused — "wait, can I use that tool? Was that my call or A's call?"

### Custom handoff tools — the fix

You can override `create_handoff_tool` to:

- **Pass only a task description** — the supervisor distills A's findings into a clean task for B.
- **Pass only filtered state fields** — share `current_task` and `findings`, hide `internal_scratchpad`.
- **Strip prior agents' tool calls** — keep messages, drop ToolMessages from other agents.

```python
# Custom handoff that passes only a task description, not the whole history
def create_clean_handoff(agent_name: str, name: str, description: str):
    @tool(name, description=description)
    def handoff(
        task_description: Annotated[str, "Clean, self-contained task for the next agent"],
        state: Annotated[dict, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId],
    ):
        tool_message = ToolMessage(
            content=f"Handing off to {agent_name}: {task_description}",
            name=name, tool_call_id=tool_call_id,
        )
        # Note: we pass ONLY the new task message + a small state slice — not full history
        return Command(
            goto=agent_name,
            graph=Command.PARENT,
            update={"messages": [tool_message], "current_task": task_description},
        )
    return handoff
```

### Decision rules for what to pass

| State item | Pass to next agent? |
|---|---|
| Original user message | Yes, always |
| Prior agents' ToolMessages | No — adds noise |
| The current task description | Yes — this is the new agent's brief |
| Findings / structured results from prior agents | Yes |
| Working scratchpad of prior agents | No — private to that agent |
| Memory retrieval results | Maybe — re-retrieve if relevant for the new context |

### Handoff back vs. handoff forward

A subtle distinction. In a supervisor:

- **Specialist → Supervisor** ("handoff back") — the specialist is done, returns control.
- **Supervisor → Next specialist** ("handoff forward") — supervisor delegates to a different agent.

The `add_handoff_back_messages` config controls whether handoff-back messages are added to state. Default is `True`. For cost-sensitive supervisors, set it to `False` — the supervisor doesn't need to re-read "transferred back to supervisor" five times.

### Handoff failure modes

| Bug | Symptom | Fix |
|---|---|---|
| Receiving agent doesn't know what to do | Re-asks the user for context | Supervisor's handoff includes an explicit task description |
| Receiving agent re-does work | Repeats research the prior agent did | Pass findings explicitly; don't rely on the model to scan history |
| Conversation loops | A → B → A → B | Track handoff count; alarm if > 3 between same pair |
| Lost context after long handoff chain | Final agent has no idea what the user originally asked | Always preserve the *original user message* across handoffs |

### A production handoff pattern

```python
# In the supervisor's "delegate to specialist" tool:
handoff_payload = {
    "original_user_message": state["messages"][0],   # the truth, never modified
    "task_for_you": clean_task_description,          # what THIS specialist should do
    "context_so_far": prior_findings,                # only structured results, no scratchpad
    "constraints": "use only the tools you have; max 5 tool calls",
}
```

That's a handoff that doesn't leak context, gives the specialist what they need, and bounds their behavior.

> **Interview note.** *"How do you keep multi-agent context manageable?"* Three answers, in order: (1) custom handoff tools that pass task descriptions, not full history. (2) `add_handoff_back_messages=False` to avoid supervisor re-reading transfer messages. (3) For very long chains, periodically summarize the running state.


In [19]:
# Custom handoff tool — pass only what the next agent needs.
# Production API uses LangChain's @tool + InjectedState + Command.
# We show the structure with a runnable simulation.

handoff_code = '''
# Production code — requires `langgraph` and `langgraph-supervisor`.
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langgraph.prebuilt import InjectedState

def create_clean_handoff_tool(*, agent_name: str, name: str, description: str):
    """Custom handoff: passes ONLY a task description and the original user message.
    Does NOT pass the supervisor's intermediate reasoning, prior tool calls, etc."""

    @tool(name, description=description)
    def handoff(
        task_description: Annotated[str, "Clean, self-contained task for the next agent"],
        state: Annotated[dict, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        # Extract only what the next agent needs
        original_user_msg = state["messages"][0] if state.get("messages") else None
        prior_findings = state.get("findings", {})

        tool_message = ToolMessage(
            content=f"Handing off to {agent_name}: {task_description}",
            name=name, tool_call_id=tool_call_id,
        )

        return Command(
            goto=agent_name,
            graph=Command.PARENT,
            update={
                "messages": [original_user_msg, tool_message] if original_user_msg else [tool_message],
                "current_task": task_description,
                "context_so_far": prior_findings,
            },
        )
    return handoff

# Use it
my_handoff = create_clean_handoff_tool(
    agent_name="writer",
    name="hand_to_writer",
    description="Hand off to the writer when research is complete. Provide a clear task description.",
)
'''
print(handoff_code)

# --- Simulation showing what gets transferred ---
print("\n=== Simulation: default vs clean handoff ===\n")

# Imagine the supervisor → specialist → supervisor → next specialist chain has accumulated state
fat_state = {
    "messages": [
        {"role": "user", "content": "Compare X and Y"},
        {"role": "assistant", "content": "Let me delegate to researcher"},
        {"role": "tool", "content": "researcher tool call 1 result"},
        {"role": "tool", "content": "researcher tool call 2 result"},
        {"role": "tool", "content": "researcher tool call 3 result"},
        {"role": "assistant", "content": "Researcher done; routing to writer"},
    ],
    "findings": {"X": "fact 1", "Y": "fact 2"},
    "internal_scratchpad": "supervisor's private notes — should NOT leak",
}

def default_handoff_payload(state: dict, target: str) -> dict:
    """The default behavior: pass everything."""
    return {"target": target, **state}

def clean_handoff_payload(state: dict, target: str, task: str) -> dict:
    """The custom behavior: pass only what the next agent needs."""
    original_user = state["messages"][0]
    return {
        "target": target,
        "messages": [original_user, {"role": "assistant", "content": f"Task: {task}"}],
        "context_so_far": state["findings"],
        # internal_scratchpad NOT included → privacy preserved
    }

default = default_handoff_payload(fat_state, "writer")
clean = clean_handoff_payload(fat_state, "writer", task="Write a 200-word comparison report")

print(f"Default handoff payload — message count: {len(default['messages'])}")
print(f"  Includes internal_scratchpad? {'internal_scratchpad' in default}  ← LEAK")
print(f"  Approx context bloat: {sum(len(str(m)) for m in default['messages'])} chars\n")

print(f"Clean handoff payload — message count: {len(clean['messages'])}")
print(f"  Includes internal_scratchpad? {'internal_scratchpad' in clean}")
print(f"  Approx context size: {sum(len(str(m)) for m in clean['messages'])} chars")
print(f"  Includes findings: {clean['context_so_far']}")
print("\nThe clean handoff passes ~1/3 the tokens AND respects state privacy.")



# Production code — requires `langgraph` and `langgraph-supervisor`.
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langgraph.prebuilt import InjectedState

def create_clean_handoff_tool(*, agent_name: str, name: str, description: str):
    """Custom handoff: passes ONLY a task description and the original user message.
    Does NOT pass the supervisor's intermediate reasoning, prior tool calls, etc."""

    @tool(name, description=description)
    def handoff(
        task_description: Annotated[str, "Clean, self-contained task for the next agent"],
        state: Annotated[dict, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        # Extract only what the next agent needs
        original_user_msg = state["messages"][0] if state.get("messages") else None
        prior_findings = state.get("findings", {})

     

---
# Synthesis & Cheat Sheet

## The agent design decision tree

You now have the vocabulary for every common decision in agent design. The order to walk through them:

```
1. Does this task need an agent at all, or just a chain?
     - Single retrieval + single LLM call → chain (use RAG, see prior notebook)
     - Needs to act, observe, adapt → agent (continue)

2. Does the task complete in <10 steps?
     - YES → single ReAct agent. (§1)
     - NO → continue.

3. Is the plan knowable upfront?
     - YES → plan-and-execute. (§2)
     - NO → ReAct or interleaved planning (write_todos). (§8)

4. Is there a clear success criterion?
     - YES → add reflection layer with max_iterations. (§2)

5. How big is the task? Project or question?
     - Question → create_agent
     - Project → DeepAgents (planning + filesystem + subagents). (§16)

6. Are there independent subtasks?
     - YES → scatter-gather; spawn subagents in parallel. (§15)

7. Are there 3+ distinct skill domains?
     - YES → supervisor (with langgraph_supervisor). (§13)
     - With 2-3 personas where user "talks to" specialists → swarm. (§14)

8. Are any actions destructive, money-moving, or external?
     - YES → HITL (LangGraph interrupts). (§9)

9. Does the agent need to remember across sessions?
     - YES → memory hierarchy: working / short-term / long-term / episodic. (§5-6)
     - Always filter long-term memory by user_id at the DB layer. (§6)

10. Will multiple users hit this concurrently?
     - YES → async server. Stream the response. Use checkpointer (Postgres) for persistence. (§10-11)
```

## The cheat sheet

### Libraries (2026 defaults)

| Purpose | Library | Notes |
|---|---|---|
| Agent runtime | `langgraph` v1.x | The graph runtime; cycles, conditional edges, checkpointers |
| Simple agent harness | `langchain.agents.create_agent` | One-line ReAct agent with middleware support |
| Multi-agent supervisor | `langgraph-supervisor` | `create_supervisor`, `create_handoff_tool` |
| Multi-agent swarm | `langgraph-swarm` | `create_swarm`, peer-to-peer handoffs |
| Batteries-included harness | `deepagents` | Planning + FS + subagents + prompt baked in |
| State persistence | `langgraph.checkpoint.postgres` (`PostgresSaver`) | Production checkpointer; SqliteSaver for local, MemorySaver for dev |
| LLM providers | `langchain-anthropic`, `langchain-openai`, `langchain-google-genai` | One package per provider |
| Tool I/O validation | `pydantic` v2 | Pydantic for API boundaries; TypedDict for LangGraph state |

### The four agent loops

| Pattern | When | Cost |
|---|---|---|
| ReAct | Dynamic, exploratory | N LLM calls |
| Plan-and-Execute | Known workflow, parallelizable | 1 expensive plan + N cheap executor calls |
| Reflection | Clear success criterion | 2× per iteration; cap iterations |
| Interleaved (write_todos) | Long-horizon, project-shaped | Marginal (the planning tool is a no-op) |

### The four multi-agent patterns

| Pattern | Library | Best for | Watch out for |
|---|---|---|---|
| Supervisor | `langgraph-supervisor` | 3-7 specialists | Bottleneck at the hub |
| Swarm | `langgraph-swarm` | Persona handoffs, low latency | Hard to debug |
| Scatter-Gather | LangGraph + asyncio | Parallel independent subtasks | Aggregation is hard |
| Hierarchical | LangGraph (custom) | 10+ specialists in teams | Latency stacks |

### The memory hierarchy

| Type | Lifetime | Store |
|---|---|---|
| Working / scratchpad | This task | Context window / state dict |
| Short-term | Session | Buffer (in-memory or Redis) |
| Long-term | Forever | Vector DB, filtered by user_id |
| Episodic | Forever | SQL table with timestamps + outcomes |

### Numbers worth remembering

| Number | What | Context |
|---|---|---|
| **15×** | Token cost of multi-agent vs single-agent | Anthropic blog — justifies multi-agent only for high-value tasks |
| **90.2%** | Improvement of Opus-lead + Sonnet-subagents over single-Opus | Anthropic research evals |
| **3-10** | Tool calls per subagent on fact-finding | Anthropic scaling rule |
| **2-4** | Subagents for comparison tasks | Anthropic scaling rule |
| **10+** | Subagents for complex research | Anthropic scaling rule |
| **25** | Default `recursion_limit` for supervisor graphs | Higher = suspicious of looping bug |
| **4K-8K tokens** | When to summarize-buffer conversation | Prevents context overflow |
| **3-5 retries** | Max for transient errors with exponential backoff | Don't retry permanent errors |

## Interview narration

### The 60-second answer to "what is a multi-agent system?"

> "A multi-agent system is multiple LLM-driven agents coordinating to handle a task that one agent can't, won't, or shouldn't handle alone. Each agent has its own prompt, its own tools, and its own loop. They coordinate through a pattern — supervisor, swarm, scatter-gather, or hierarchical. The reason to use one is rarely 'because it's smarter' and usually 'because it parallelizes work', 'because specialization improves tool-selection accuracy', or 'because context isolation prevents one task from polluting another.' Anthropic measured ~15× more tokens vs single-agent, so you only reach for it when the value of the task clearly exceeds the extra cost."

### The 180-second answer for an interview

Add to the above:

> "The four patterns are supervisor — a central agent routes between specialists, easy to reason about; swarm — agents hand off peer-to-peer through handoff tools, lower latency; scatter-gather — a planner spawns N parallel workers and a gatherer aggregates, used by Anthropic for their research system; and hierarchical — supervisors of supervisors, for 10+ specialists.
>
> For state management, every agent in a real system needs four memory types: working memory in the context window, short-term as a conversation buffer (with summary-buffer once over 4K tokens), long-term as a per-user vector store with strict user_id filtering at the DB layer, and episodic as structured records of past events for Reflexion-style learning.
>
> For long-horizon tasks, I use DeepAgents — it's the open-source extraction of Claude Code's architecture, with planning via `write_todos`, a filesystem as the agent's extended memory, subagents for context isolation, and a detailed system prompt. Three lines to a research-grade agent.
>
> For human-in-the-loop, LangGraph `interrupt` plus a Postgres checkpointer — pause for approval, persist state, resume hours later when the human approves. I use this for destructive actions, money movement, or external messaging.
>
> For deployment, async server, streaming for perceived latency, background jobs for tasks over 30 seconds, and aggressive use of checkpointing for resumability."

That's a 180-second answer that hits foundations, multi-agent patterns, memory, DeepAgents, HITL, and serving. Use it.

### Connecting to your past work (Samarth-specific)

For your Microsoft Seller Copilot / Invoice Copilot work, the bridge is **specialized agents with tool retrieval**. You can say:

> "On Seller Copilot we faced the same tool-selection-at-scale problem that hits every multi-agent system once tools grow past a dozen — we used tool retrieval to keep the candidate set small per query. When I think about supervisor-pattern agents, I think about the same problem: don't show the model 50 tools, show it the 5 that matter for the current routing decision."

---

*End of Notebook 1.*

**Next: N2 covers protocols (MCP, A2A) and coding agents (Claude Code / DeepAgents Code / Antigravity patterns).**  
**Then: N3 covers evaluation, serving, monitoring, guardrails, and the maintain-an-agent loop.**
